In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd

def list_gpkg_layers(gpkg_path):
    try:
        layers_df = gpd.list_layers(gpkg_path)
        return layers_df["name"].tolist()
    except Exception as e:
        print(f"Не удалось получить список слоев для {gpkg_path}: {e}")
        return []

INPUT_GPKG = "moscow_realestate_before.gpkg"

if not os.path.exists(INPUT_GPKG):
    raise FileNotFoundError(
        f"Не найден исходный файл {INPUT_GPKG}."
    )

print("Слои в GPKG:")
input_layers = list_gpkg_layers(INPUT_GPKG)
print(input_layers if input_layers else "Список слоев не получен")

INPUT_LAYER = input_layers[0] if input_layers else None

OUTPUT_GPKG = "apartments_prepared.gpkg"
OUTPUT_LAYER = "apartments_prepared"
OUTPUT_CSV = "apartments_prepared.csv"

SOURCE_CRS = "EPSG:4326"
TARGET_CRS = "EPSG:32637"

PRICE_M2_LOW_Q = 0.01
PRICE_M2_HIGH_Q = 0.99

MIN_AREA = 15
MAX_AREA = 300

MAX_ROOMS = 8
MAX_FLOORS = 100

CURRENT_YEAR = 2026

gdf = gpd.read_file(INPUT_GPKG, layer=INPUT_LAYER)

print("Исходное количество строк:", len(gdf))
print("CRS исходного слоя:", gdf.crs)
print("Колонки:")
print(gdf.columns.tolist())

if "geometry" not in gdf.columns or gdf.geometry.isna().all():
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf["longitude"], gdf["latitude"]),
        crs=SOURCE_CRS
    )

if gdf.crs is None:
    gdf = gdf.set_crs(SOURCE_CRS)

required_cols = [
    "id",
    "price",
    "url",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "address",
    "latitude",
    "longitude",
    "building_year",
    "material_type"
]

missing_cols = [col for col in required_cols if col not in gdf.columns]

if missing_cols:
    raise ValueError(f"В таблице не хватает колонок: {missing_cols}")

num_cols = [
    "price",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "latitude",
    "longitude"
]

for col in num_cols:
    gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

n_before_technical = len(gdf)

gdf = gdf[
    (gdf["price"].notna()) &
    (gdf["total_area"].notna()) &
    (gdf["latitude"].notna()) &
    (gdf["longitude"].notna()) &
    (gdf["price"] > 0) &
    (gdf["total_area"] > 0)
].copy()

gdf.loc[gdf["floor_number"] <= 0, "floor_number"] = np.nan
gdf.loc[gdf["floors_count"] <= 0, "floors_count"] = np.nan
gdf.loc[gdf["floor_number"] > gdf["floors_count"], "floor_number"] = np.nan

gdf.loc[gdf["rooms_count"] < 0, "rooms_count"] = np.nan

n_after_technical = len(gdf)

print("\nТехническая чистка:")
print("Было строк:", n_before_technical)
print("Осталось строк:", n_after_technical)
print("Удалено:", n_before_technical - n_after_technical)

import re

MIN_AREA_PER_ROOM = 12

def parse_share_value(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    if s == "" or s in ["nan", "none", "null"]:
        return np.nan

    s = s.replace(",", ".")

    if "%" in s:
        s_num = re.sub(r"[^0-9.]", "", s)
        try:
            return float(s_num) / 100
        except:
            return np.nan

    if "/" in s:
        parts = s.split("/")
        if len(parts) == 2:
            try:
                numerator = float(parts[0])
                denominator = float(parts[1])
                if denominator != 0:
                    return numerator / denominator
            except:
                return np.nan

    try:
        return float(s)
    except:
        return np.nan

n_before_shares = len(gdf)

if "share_value" in gdf.columns:
    gdf["share_value_raw"] = gdf["share_value"]

    gdf["share_value_str"] = (
        gdf["share_value"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    gdf["share_value_filled"] = ~(
        gdf["share_value"].isna() |
        gdf["share_value_str"].isin(["", "nan", "none", "null"])
    )

    gdf["share_value_num"] = gdf["share_value"].apply(parse_share_value)

    full_apartment_mask = (
        (~gdf["share_value_filled"]) |
        (np.isclose(gdf["share_value_num"], 1.0))
    )

    gdf = gdf[full_apartment_mask].copy()

gdf["area_per_room_tmp"] = np.where(
    gdf["rooms_count"] > 0,
    gdf["total_area"] / gdf["rooms_count"],
    np.nan
)

bad_share_like_mask = (
    gdf["rooms_count"].notna() &
    (gdf["rooms_count"] >= 2) &
    (gdf["area_per_room_tmp"] < MIN_AREA_PER_ROOM)
)

print("\nПодозрительных объектов по area_per_room:", bad_share_like_mask.sum())

gdf = gdf[~bad_share_like_mask].copy()
gdf = gdf.drop(columns=["area_per_room_tmp"])

n_after_shares = len(gdf)

print("\nУдаление долей / комнат:")
print("Было строк:", n_before_shares)
print("Осталось строк:", n_after_shares)
print("Удалено:", n_before_shares - n_after_shares)
print(
    "Доля удаленных:",
    round((n_before_shares - n_after_shares) / n_before_shares * 100, 2),
    "%"
)

gdf["price_m2"] = gdf["price"] / gdf["total_area"]

gdf["ln_price"] = np.log(gdf["price"])
gdf["ln_price_m2"] = np.log(gdf["price_m2"])

gdf["ln_total_area"] = np.log(gdf["total_area"])

gdf["area_per_room"] = np.where(
    gdf["rooms_count"] > 0,
    gdf["total_area"] / gdf["rooms_count"],
    np.nan
)

gdf["rooms_group"] = np.select(
    [
        gdf["rooms_count"] == 0,
        gdf["rooms_count"] == 1,
        gdf["rooms_count"] == 2,
        gdf["rooms_count"] == 3,
        gdf["rooms_count"] >= 4
    ],
    [
        "studio",
        "1_room",
        "2_rooms",
        "3_rooms",
        "4plus_rooms"
    ],
    default=None
)

gdf["floor_ratio"] = gdf["floor_number"] / gdf["floors_count"]

gdf["is_first_floor"] = np.where(gdf["floor_number"] == 1, 1, 0)

gdf["is_last_floor"] = np.where(
    gdf["floor_number"] == gdf["floors_count"],
    1,
    0
)

gdf["is_middle_floor"] = np.where(
    (gdf["floor_number"] > 1) &
    (gdf["floor_number"] < gdf["floors_count"]),
    1,
    0
)

floor_missing = gdf["floor_number"].isna() | gdf["floors_count"].isna()

gdf.loc[
    floor_missing,
    ["floor_ratio", "is_first_floor", "is_last_floor", "is_middle_floor"]
] = np.nan

gdf["floor_type"] = np.select(
    [
        gdf["is_first_floor"] == 1,
        gdf["is_last_floor"] == 1,
        gdf["is_middle_floor"] == 1
    ],
    [
        "first",
        "last",
        "middle"
    ],
    default=None
)

gdf["building_age"] = CURRENT_YEAR - gdf["building_year"]

gdf.loc[gdf["building_year"] < 1800, "building_age"] = np.nan
gdf.loc[gdf["building_year"] > CURRENT_YEAR, "building_age"] = np.nan

gdf["building_age_sq"] = gdf["building_age"] ** 2

n_before_outliers = len(gdf)

q_low = gdf["price_m2"].quantile(PRICE_M2_LOW_Q)
q_high = gdf["price_m2"].quantile(PRICE_M2_HIGH_Q)

gdf = gdf[
    (gdf["price_m2"] >= q_low) &
    (gdf["price_m2"] <= q_high)
].copy()

gdf = gdf[
    (gdf["total_area"] >= MIN_AREA) &
    (gdf["total_area"] <= MAX_AREA)
].copy()

gdf = gdf[
    (gdf["rooms_count"].isna()) |
    ((gdf["rooms_count"] >= 0) & (gdf["rooms_count"] <= MAX_ROOMS))
].copy()

gdf = gdf[
    (gdf["floors_count"].isna()) |
    ((gdf["floors_count"] >= 1) & (gdf["floors_count"] <= MAX_FLOORS))
].copy()

gdf = gdf[
    (gdf["floor_ratio"].isna()) |
    ((gdf["floor_ratio"] > 0) & (gdf["floor_ratio"] <= 1))
].copy()

n_after_outliers = len(gdf)

print("\nЧистка выбросов:")
print("Было строк:", n_before_outliers)
print("Осталось строк:", n_after_outliers)
print("Удалено:", n_before_outliers - n_after_outliers)
print(
    "Доля удаленных:",
    round((n_before_outliers - n_after_outliers) / n_before_outliers * 100, 2),
    "%"
)

print("\nГраницы price_m2:")
print("Нижняя граница:", round(q_low, 2))
print("Верхняя граница:", round(q_high, 2))

gdf["material_type_clean"] = (
    gdf["material_type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

material_dummies = pd.get_dummies(
    gdf["material_type_clean"],
    prefix="material",
    dummy_na=False
)

gdf = pd.concat([gdf, material_dummies], axis=1)

gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=gdf.crs)

gdf = gdf.to_crs(TARGET_CRS)

print("\nФинальное количество квартир:", len(gdf))
print("Финальный CRS:", gdf.crs)

print("\nОписание price_m2:")
print(gdf["price_m2"].describe())

print("\nОписание total_area:")
print(gdf["total_area"].describe())

print("\nОписание rooms_count:")
print(gdf["rooms_count"].describe())

print("\nОписание floor_ratio:")
print(gdf["floor_ratio"].describe())

if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

gdf.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG"
)

gdf.drop(columns="geometry").to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nГотово.")
print(f"GPKG сохранен: {OUTPUT_GPKG}, слой: {OUTPUT_LAYER}")
print(f"CSV сохранен: {OUTPUT_CSV}")


Слои в GPKG:
['moscow_realestate_1__merged']
Исходное количество строк: 33536
CRS исходного слоя: EPSG:32637
Колонки:
['id', 'price', 'url', 'total_area', 'rooms_count', 'floor_number', 'floors_count', 'address', 'latitude', 'longitude', 'building_year', 'material_type', 'share_value', 'layer', 'path', 'geometry']

Техническая чистка:
Было строк: 33536
Осталось строк: 33536
Удалено: 0

Подозрительных объектов по area_per_room: 69

Удаление долей / комнат:
Было строк: 33536
Осталось строк: 32984
Удалено: 552
Доля удаленных: 1.65 %

Чистка выбросов:
Было строк: 32984
Осталось строк: 31614
Удалено: 1370
Доля удаленных: 4.15 %

Границы price_m2:
Нижняя граница: 75000.0
Верхняя граница: 2182029.31

Финальное количество квартир: 31614
Финальный CRS: EPSG:32637

Описание price_m2:
count    3.161400e+04
mean     5.056260e+05
std      2.926250e+05
min      7.500000e+04
25%      3.251149e+05
50%      4.182911e+05
75%      5.816044e+05
max      2.181818e+06
Name: price_m2, dtype: float64

Описани

In [ ]:
import os
import geopandas as gpd

INPUT_GPKG = "apartments_prepared.gpkg"
INPUT_LAYER = "apartments_prepared"

OUTPUT_GPKG = "apartments_model_base.gpkg"
OUTPUT_LAYER = "apartments_model_base"
OUTPUT_CSV = "apartments_model_base.csv"

if not os.path.exists(INPUT_GPKG):
    raise FileNotFoundError(
        f"Не найден промежуточный файл {INPUT_GPKG}."
    )

apartments = gpd.read_file(
    INPUT_GPKG,
    layer=INPUT_LAYER
)

print("Исходное количество квартир:", len(apartments))
print("CRS:", apartments.crs)
print("Колонки:")
print(apartments.columns.tolist())

required_cols = [
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "material_type",
    "geometry"
]

missing_cols = [
    col for col in required_cols
    if col not in apartments.columns
]

if missing_cols:
    raise ValueError(f"Не хватает колонок: {missing_cols}")

model_base_cols = [
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "material_type",
    "geometry"
]

optional_check_cols = [
    "url",
    "address",
    "latitude",
    "longitude",
    "ln_price",
    "ln_total_area",
    "area_per_room",
    "floor_ratio",
    "is_first_floor",
    "is_last_floor",
    "is_middle_floor",
    "building_age",
    "building_age_sq",
    "rooms_group",
    "floor_type"
]

existing_optional_cols = [
    col for col in optional_check_cols
    if col in apartments.columns
]

final_cols = [
    col for col in model_base_cols
    if col != "geometry"
] + existing_optional_cols + ["geometry"]

apartments_base = apartments[final_cols].copy()

print("\nИтоговое количество квартир:", len(apartments_base))
print("Итоговые колонки:")
print(apartments_base.columns.tolist())

print("\nПервые строки:")
print(
    apartments_base[
        [
            "id",
            "price",
            "price_m2",
            "ln_price_m2",
            "total_area",
            "rooms_count",
            "floor_number",
            "floors_count",
            "building_year",
            "material_type"
        ]
    ].head()
)

if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

apartments_base.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG"
)

apartments_base.drop(columns="geometry").to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nГотово.")
print(f"GPKG сохранен: {OUTPUT_GPKG}, слой: {OUTPUT_LAYER}")
print(f"CSV сохранен: {OUTPUT_CSV}")


Исходное количество квартир: 31614
CRS: EPSG:32637
Колонки:
['id', 'price', 'url', 'total_area', 'rooms_count', 'floor_number', 'floors_count', 'address', 'latitude', 'longitude', 'building_year', 'material_type', 'share_value', 'layer', 'path', 'share_value_raw', 'share_value_str', 'share_value_filled', 'share_value_num', 'price_m2', 'ln_price', 'ln_price_m2', 'ln_total_area', 'area_per_room', 'rooms_group', 'floor_ratio', 'is_first_floor', 'is_last_floor', 'is_middle_floor', 'floor_type', 'building_age', 'building_age_sq', 'material_type_clean', 'material_aerocreteblock', 'material_block', 'material_brick', 'material_monolith', 'material_monolithbrick', 'material_none', 'material_old', 'material_panel', 'material_stalin', 'material_wood', 'geometry']

Итоговое количество квартир: 31614
Итоговые колонки:
['id', 'price', 'price_m2', 'ln_price_m2', 'total_area', 'rooms_count', 'floor_number', 'floors_count', 'building_year', 'material_type', 'url', 'address', 'latitude', 'longitude', 'l

In [ ]:
import os
import numpy as np
import geopandas as gpd

def list_gpkg_layers(gpkg_path):
    try:
        layers_df = gpd.list_layers(gpkg_path)
        return layers_df["name"].tolist()
    except Exception as e:
        print(f"Не удалось получить список слоев для {gpkg_path}: {e}")
        return []

APARTMENTS_GPKG = "apartments_model_base.gpkg"
APARTMENTS_LAYER = "apartments_model_base"

HEX_GPKG_CANDIDATES = [
    "moscow_hex_zones_two_variants_activity_4blocks_FINAL-2.gpkg",
    "/mnt/data/moscow_hex_zones_two_variants_activity_4blocks_FINAL-2.gpkg",
]

HEX_GPKG = next(
    (path for path in HEX_GPKG_CANDIDATES if os.path.exists(path)),
    HEX_GPKG_CANDIDATES[0]
)

HEX_LAYER = "zones_two_variants_final"
HEX_ZONE_COLUMN = "zone_final_4blocks"

OUTPUT_GPKG = "apartments_with_zones.gpkg"
OUTPUT_LAYER = "apartments_with_zones"
OUTPUT_CSV = "apartments_with_zones.csv"

if not os.path.exists(APARTMENTS_GPKG):
    raise FileNotFoundError(
        f"Не найден промежуточный файл {APARTMENTS_GPKG}."
    )

if not os.path.exists(HEX_GPKG):
    raise FileNotFoundError(
        f"Не найден файл зонирования {HEX_GPKG}."
    )

def harmonize_zone_layer(hexes):
    hexes = hexes.copy()

    if "hex_id" not in hexes.columns:
        if "id" in hexes.columns:
            hexes["hex_id"] = hexes["id"]
        else:
            hexes["hex_id"] = np.arange(len(hexes))

    if HEX_ZONE_COLUMN in hexes.columns:
        hexes["zone_name"] = hexes[HEX_ZONE_COLUMN]
    elif "zone_final" in hexes.columns:
        hexes["zone_name"] = hexes["zone_final"]
        print("поле zone_final_4blocks не найдено")
    elif "zone_name" in hexes.columns:
        hexes["zone_name"] = hexes["zone_name"]
        print("поле zone_final_4blocks не найдено")
    else:
        raise ValueError(
            "В слое гексагонов нет zone_final_4blocks, zone_final или zone_name. "
            f"Колонки слоя: {hexes.columns.tolist()}"
        )

    hexes["zone_name"] = (
        hexes["zone_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    zone_aliases = {
        "центр": "core",
        "core": "core",
        "ядро": "core",
        "main_core": "core",
        "semi_periphery": "semi_periphery",
        "semiperiphery": "semi_periphery",
        "полупериферия": "semi_periphery",
        "intermediate": "semi_periphery",
        "intermediate_areas": "semi_periphery",
        "periphery": "periphery",
        "периферия": "periphery",
        "non_urban": "non_urban",
        "nonurban": "non_urban",
        "nan": np.nan,
        "none": np.nan,
    }
    hexes["zone_name"] = hexes["zone_name"].replace(zone_aliases)

    zone_rank_map = {
        "periphery": 0,
        "semi_periphery": 1,
        "core": 2,
    }
    hexes["zone_rank"] = hexes["zone_name"].map(zone_rank_map)

    print("\nРаспределение гексагонов после приведения нового зонирования:")
    print(hexes["zone_name"].value_counts(dropna=False))

    unexpected = sorted(
        set(hexes["zone_name"].dropna().unique())
        - {"core", "semi_periphery", "periphery", "non_urban"}
    )
    if unexpected:
        print("\n неожиданные названия зон:", unexpected)

    return hexes

def drop_old_zone_columns(gdf):
    old_zone_cols = [
        "hex_id",
        "zone_name",
        "zone_rank",
        "zone_core",
        "zone_semi_periphery",
    ]
    cols_to_drop = [col for col in old_zone_cols if col in gdf.columns]

    if cols_to_drop:
        print(cols_to_drop)
        gdf = gdf.drop(columns=cols_to_drop)

    return gdf

print("Слои в файле квартир:")
print(list_gpkg_layers(APARTMENTS_GPKG))

print("\nСлои в файле гексагонов:")
print(list_gpkg_layers(HEX_GPKG))

apartments = gpd.read_file(
    APARTMENTS_GPKG,
    layer=APARTMENTS_LAYER
)

hexes = gpd.read_file(
    HEX_GPKG,
    layer=HEX_LAYER
)

hexes = harmonize_zone_layer(hexes)
apartments = drop_old_zone_columns(apartments)

print("\nКвартир:", len(apartments))
print("CRS квартир:", apartments.crs)

print("\nГексагонов:", len(hexes))
print("CRS гексагонов:", hexes.crs)

print("\nКолонки квартир:")
print(apartments.columns.tolist())

print("\nКолонки гексагонов:")
print(hexes.columns.tolist())

required_apartment_cols = [
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "material_type",
    "geometry"
]

required_hex_cols = [
    "hex_id",
    "zone_name",
    "zone_rank",
    "geometry"
]

missing_apartment_cols = [
    col for col in required_apartment_cols
    if col not in apartments.columns
]

missing_hex_cols = [
    col for col in required_hex_cols
    if col not in hexes.columns
]

if missing_apartment_cols:
    raise ValueError(f"В слое квартир не хватает колонок: {missing_apartment_cols}")

if missing_hex_cols:
    raise ValueError(f"В слое гексагонов не хватает колонок: {missing_hex_cols}")

hexes = hexes.to_crs(apartments.crs)

print("\nCRS после приведения:")
print("Квартиры:", apartments.crs)
print("Гексагоны:", hexes.crs)

hexes_small = hexes[
    [
        "hex_id",
        "zone_name",
        "zone_rank",
        "geometry"
    ]
].copy()

hexes_small["zone_name"] = (
    hexes_small["zone_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("\nРаспределение гексагонов по зонам:")
print(hexes_small["zone_name"].value_counts(dropna=False))

print("\nПроверка zone_rank:")
print(
    hexes_small[
        ["zone_name", "zone_rank"]
    ]
    .drop_duplicates()
    .sort_values("zone_rank")
)

apartments_zoned = gpd.sjoin(
    apartments,
    hexes_small,
    how="left",
    predicate="within"
)

if "index_right" in apartments_zoned.columns:
    apartments_zoned = apartments_zoned.drop(columns=["index_right"])

apartments_zoned["zone_core"] = np.where(
    apartments_zoned["zone_name"] == "core",
    1,
    0
)

apartments_zoned["zone_semi_periphery"] = np.where(
    apartments_zoned["zone_name"] == "semi_periphery",
    1,
    0
)

zone_missing = apartments_zoned["zone_name"].isna()

apartments_zoned.loc[
    zone_missing,
    ["zone_rank", "zone_core", "zone_semi_periphery"]
] = np.nan

print("\nКвартир после spatial join:", len(apartments_zoned))

print("\nРаспределение квартир по зонам:")
print(apartments_zoned["zone_name"].value_counts(dropna=False))

missing_zone_count = apartments_zoned["zone_name"].isna().sum()
missing_zone_share = apartments_zoned["zone_name"].isna().mean() * 100

print("\nКвартир без зоны:", missing_zone_count)
print("Доля квартир без зоны:", round(missing_zone_share, 2), "%")

print("\nПроверка кодировки зон:")
print(
    apartments_zoned[
        [
            "zone_name",
            "zone_rank",
            "zone_core",
            "zone_semi_periphery"
        ]
    ]
    .drop_duplicates()
    .sort_values(["zone_rank", "zone_name"])
)

n_before_filter = len(apartments_zoned)

apartments_zoned = apartments_zoned[
    apartments_zoned["zone_name"].isin(
        ["periphery", "semi_periphery", "core"]
    )
].copy()

n_after_filter = len(apartments_zoned)

print("\nФильтр по зонам:")
print("Было:", n_before_filter)
print("Осталось:", n_after_filter)
print("Удалено:", n_before_filter - n_after_filter)

main_cols = [
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "material_type",
    "hex_id",
    "zone_name",
    "zone_rank",
    "zone_core",
    "zone_semi_periphery",
    "geometry"
]

extra_cols = [
    col for col in apartments_zoned.columns
    if col not in main_cols
]

ordered_cols = [
    col for col in main_cols
    if col != "geometry"
] + extra_cols + ["geometry"]

apartments_zoned = apartments_zoned[ordered_cols].copy()

if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

apartments_zoned.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG"
)

apartments_zoned.drop(columns="geometry").to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nГотово.")
print(f"GPKG сохранен: {OUTPUT_GPKG}, слой: {OUTPUT_LAYER}")
print(f"CSV сохранен: {OUTPUT_CSV}")


Слои в файле квартир:
['apartments_model_base']

Слои в файле гексагонов:
['zones_two_variants_final']

Распределение гексагонов после приведения нового зонирования:
zone_name
periphery         989
semi_periphery    325
core               99
non_urban          54
Name: count, dtype: int64

Квартир: 31614
CRS квартир: EPSG:32637

Гексагонов: 1467
CRS гексагонов: EPSG:32637

Колонки квартир:
['apt_uid', 'id', 'price', 'price_m2', 'ln_price_m2', 'total_area', 'rooms_count', 'floor_number', 'floors_count', 'building_year', 'material_type', 'url', 'address', 'latitude', 'longitude', 'ln_price', 'ln_total_area', 'area_per_room', 'floor_ratio', 'is_first_floor', 'is_last_floor', 'is_middle_floor', 'building_age', 'building_age_sq', 'rooms_group', 'floor_type', 'geometry']

Колонки гексагонов:
['id', 'left', 'top', 'right', 'bottom', 'row_index', 'col_index', 'hex_id', 'manual_class', 'manual_class_norm', 'is_non_urban', 'is_model_area', 'L_social', 'L_services', 'L_transport', 'L_activity', '

In [ ]:
import os
import geopandas as gpd

if not os.path.exists("apartments_prepared.gpkg"):
    raise FileNotFoundError("Не найден apartments_prepared.gpkg.")

prepared = gpd.read_file(
    "apartments_prepared.gpkg",
    layer="apartments_prepared"
)

print("Квартир в apartments_prepared:", len(prepared))
print(prepared[["price_m2", "total_area", "rooms_count", "area_per_room"]].describe())


Квартир в apartments_prepared: 31614
           price_m2    total_area   rooms_count  area_per_room
count  3.161400e+04  31614.000000  29288.000000   29288.000000
mean   5.056260e+05     73.315329      2.472753      31.164758
std    2.926250e+05     46.315585      1.095817       9.573635
min    7.500000e+04     15.000000      1.000000      12.500000
25%    3.251149e+05     42.800000      2.000000      24.333333
50%    4.182911e+05     60.000000      2.000000      29.100000
75%    5.816044e+05     85.000000      3.000000      36.600000
max    2.181818e+06    300.000000      6.000000     149.000000


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd

def list_gpkg_layers(gpkg_path):
    try:
        layers_df = gpd.list_layers(gpkg_path)
        return layers_df["name"].tolist()
    except Exception as e:
        print(f"Не удалось получить список слоев для {gpkg_path}: {e}")
        return []

warnings.filterwarnings("ignore")

TARGET_CRS = "EPSG:32637"
SOURCE_CRS_IF_MISSING = "EPSG:4326"

APARTMENTS_PREPARED_GPKG = "apartments_prepared.gpkg"
APARTMENTS_PREPARED_LAYER = "apartments_prepared"

HEX_GPKG_CANDIDATES = [
    "moscow_hex_zones_two_variants_activity_4blocks_FINAL-2.gpkg",
    "/mnt/data/moscow_hex_zones_two_variants_activity_4blocks_FINAL-2.gpkg",
]

HEX_GPKG = next(
    (path for path in HEX_GPKG_CANDIDATES if os.path.exists(path)),
    HEX_GPKG_CANDIDATES[0]
)

HEX_LAYER = "zones_two_variants_final"
HEX_ZONE_COLUMN = "zone_final_4blocks"

APARTMENTS_MODEL_BASE_GPKG = "apartments_model_base.gpkg"
APARTMENTS_MODEL_BASE_LAYER = "apartments_model_base"

APARTMENTS_WITH_ZONES_GPKG = "apartments_with_zones.gpkg"
APARTMENTS_WITH_ZONES_LAYER = "apartments_with_zones"

OUTPUT_GPKG = "apartments_hedonic_features.gpkg"
OUTPUT_LAYER = "apartments_hedonic_features"
OUTPUT_CSV = "apartments_hedonic_features.csv"

PROCESSED_LAYERS_GPKG = "processed_external_layers.gpkg"
SAVE_PROCESSED_LAYERS = True

LAYER_PATHS = {

    "mcd": {
        "gpkg": "moscow_mcd.gpkg",
        "layer": None
    },
    "metro_mcc": {
        "gpkg": "moscow_metro_mck.gpkg",
        "layer": None
    },
    "stops": {
        "gpkg": "public_transport_stops.gpkg",
        "layer": None
    },

    "social": {
        "gpkg": "moscow_social_infr.gpkg",
        "layer": None
    },
    "parks": {
        "gpkg": "moscow_parks.gpkg",
        "layer": None
    },
    "rivers": {
        "gpkg": "moscow_rivers.gpkg",
        "layer": None
    },

    "highway": {
        "gpkg": "moscow_highway.gpkg",
        "layer": None
    },
    "railway": {
        "gpkg": "moscow_railway.gpkg",
        "layer": None
    },
    "industrial": {
        "gpkg": "moscow_industrial_areas_filtered.gpkg",
        "layer": None
    },
    "aerodromes": {
        "gpkg": "moscow_aerodromes.gpkg",
        "layer": None
    },
    "waste": {
        "gpkg": "moscow_waste_facilities.gpkg",
        "layer": None
    },
    "cemeteries": {
        "gpkg": "moscow_cemeteries.gpkg",
        "layer": None
    },
}

def pick_layer(gpkg_path, preferred_layer=None):
    layers = list_gpkg_layers(gpkg_path)

    if preferred_layer is not None:
        if layers and preferred_layer not in layers:
            raise ValueError(
                f"В файле {gpkg_path} нет слоя '{preferred_layer}'. "
                f"Доступные слои: {layers}"
            )
        return preferred_layer

    if len(layers) > 0:
        return layers[0]

    print(f"Список слоев для {gpkg_path} не получен, будет прочитан первый слой GPKG.")
    return None

def read_gpkg_auto(gpkg_path, layer=None, label=None):
    if not os.path.exists(gpkg_path):
        raise FileNotFoundError(f"Не найден файл: {gpkg_path}")

    layer_to_read = pick_layer(gpkg_path, layer)

    gdf = gpd.read_file(gpkg_path, layer=layer_to_read)

    print("\n" + "=" * 70)
    print(f"Загружен слой: {label if label else gpkg_path}")
    print("Файл:", gpkg_path)
    print("Слой:", layer_to_read)
    print("Количество объектов:", len(gdf))
    print("CRS:", gdf.crs)
    print("Типы геометрии:")
    print(gdf.geom_type.value_counts(dropna=False))

    return gdf

def harmonize_zone_layer(hexes):
    hexes = hexes.copy()

    if "hex_id" not in hexes.columns:
        if "id" in hexes.columns:
            hexes["hex_id"] = hexes["id"]
        else:
            hexes["hex_id"] = np.arange(len(hexes))

    if HEX_ZONE_COLUMN in hexes.columns:
        hexes["zone_name"] = hexes[HEX_ZONE_COLUMN]
    elif "zone_final" in hexes.columns:
        hexes["zone_name"] = hexes["zone_final"]
        print("поле zone_final_4blocks не найдено")
    elif "zone_name" in hexes.columns:
        hexes["zone_name"] = hexes["zone_name"]
        print("поле zone_final_4blocks не найдено")
    else:
        raise ValueError(
            "В слое гексагонов нет zone_final_4blocks, zone_final или zone_name. "
            f"Колонки слоя: {hexes.columns.tolist()}"
        )

    hexes["zone_name"] = (
        hexes["zone_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    zone_aliases = {
        "центр": "core",
        "core": "core",
        "ядро": "core",
        "main_core": "core",
        "semi_periphery": "semi_periphery",
        "semiperiphery": "semi_periphery",
        "полупериферия": "semi_periphery",
        "intermediate": "semi_periphery",
        "intermediate_areas": "semi_periphery",
        "periphery": "periphery",
        "периферия": "periphery",
        "non_urban": "non_urban",
        "nonurban": "non_urban",
        "nan": np.nan,
        "none": np.nan,
    }
    hexes["zone_name"] = hexes["zone_name"].replace(zone_aliases)

    zone_rank_map = {
        "periphery": 0,
        "semi_periphery": 1,
        "core": 2,
    }
    hexes["zone_rank"] = hexes["zone_name"].map(zone_rank_map)

    print("\nРаспределение гексагонов после приведения нового зонирования:")
    print(hexes["zone_name"].value_counts(dropna=False))

    unexpected = sorted(
        set(hexes["zone_name"].dropna().unique())
        - {"core", "semi_periphery", "periphery", "non_urban"}
    )
    if unexpected:
        print("\n найдены неожиданные названия зон:", unexpected)

    return hexes

def drop_old_zone_columns(gdf):
    old_zone_cols = [
        "hex_id",
        "zone_name",
        "zone_rank",
        "zone_core",
        "zone_semi_periphery",
    ]
    cols_to_drop = [col for col in old_zone_cols if col in gdf.columns]

    if cols_to_drop:
        print(cols_to_drop)
        gdf = gdf.drop(columns=cols_to_drop)

    return gdf

def guess_and_set_crs(gdf, target_crs=TARGET_CRS):
    if gdf.crs is not None:
        return gdf

    bounds = gdf.total_bounds
    minx, miny, maxx, maxy = bounds

    if -180 <= minx <= 180 and -180 <= maxx <= 180 and -90 <= miny <= 90 and -90 <= maxy <= 90:
        gdf = gdf.set_crs(SOURCE_CRS_IF_MISSING)
        print("CRS отсутствовал. Установлен EPSG:4326.")
    else:
        gdf = gdf.set_crs(target_crs)
        print(f"CRS отсутствовал. Установлен {target_crs}.")

    return gdf

def prepare_layer(gdf, target_crs=TARGET_CRS, make_valid=True):
    gdf = gdf.copy()

    if "geometry" not in gdf.columns:
        raise ValueError("В слое нет geometry.")

    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    gdf = guess_and_set_crs(gdf, target_crs=target_crs)
    gdf = gdf.to_crs(target_crs)

    if make_valid:
        try:
            gdf["geometry"] = gdf.geometry.make_valid()
        except Exception:

            try:
                gdf["geometry"] = gdf.geometry.buffer(0)
            except Exception:
                pass

    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    try:
        gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    except TypeError:
        gdf = gdf.explode().reset_index(drop=True)

    return gdf

def save_layer(gdf, gpkg_path, layer_name, remove_file=False):
    if remove_file and os.path.exists(gpkg_path):
        os.remove(gpkg_path)

    gdf.to_file(
        gpkg_path,
        layer=layer_name,
        driver="GPKG"
    )

def add_nearest_distance(apartments_gdf, feature_gdf, output_col):
    apartments_gdf = apartments_gdf.copy()

    if feature_gdf is None or len(feature_gdf) == 0:
        apartments_gdf[output_col] = np.nan
        print(f"{output_col}: слой пустой, записан NaN.")
        return apartments_gdf

    left = apartments_gdf[["apt_uid", "geometry"]].copy()
    right = feature_gdf[["geometry"]].copy()

    nearest = gpd.sjoin_nearest(
        left,
        right,
        how="left",
        distance_col=output_col
    )

    dist = (
        nearest
        .groupby("apt_uid")[output_col]
        .min()
        .reset_index()
    )

    apartments_gdf = apartments_gdf.merge(
        dist,
        on="apt_uid",
        how="left"
    )

    print(f"{output_col}: добавлено.")
    return apartments_gdf

def add_dummy_from_distance(apartments_gdf, distance_col, threshold_m, output_col):
    apartments_gdf[output_col] = np.where(
        apartments_gdf[distance_col].notna() &
        (apartments_gdf[distance_col] <= threshold_m),
        1,
        0
    ).astype(int)

    print(f"{output_col}: создано по {distance_col} <= {threshold_m} м.")
    return apartments_gdf

def add_points_count_within_radius(apartments_gdf, points_gdf, radius_m, output_col):
    apartments_gdf = apartments_gdf.copy()

    if points_gdf is None or len(points_gdf) == 0:
        apartments_gdf[output_col] = 0
        print(f"{output_col}: слой пустой, записан 0.")
        return apartments_gdf

    buffers = apartments_gdf[["apt_uid", "geometry"]].copy()
    buffers["geometry"] = buffers.geometry.buffer(radius_m)

    joined = gpd.sjoin(
        buffers,
        points_gdf[["geometry"]],
        how="left",
        predicate="contains"
    )

    counts = (
        joined
        .dropna(subset=["index_right"])
        .groupby("apt_uid")
        .size()
        .reset_index(name=output_col)
    )

    apartments_gdf = apartments_gdf.merge(
        counts,
        on="apt_uid",
        how="left"
    )

    apartments_gdf[output_col] = apartments_gdf[output_col].fillna(0).astype(int)

    print(f"{output_col}: посчитано в радиусе {radius_m} м.")
    return apartments_gdf

def add_polygon_area_within_radius(apartments_gdf, polygons_gdf, radius_m, output_col):
    apartments_gdf = apartments_gdf.copy()

    if polygons_gdf is None or len(polygons_gdf) == 0:
        apartments_gdf[output_col] = 0.0
        print(f"{output_col}: слой пустой, записан 0.")
        return apartments_gdf

    polygons_gdf = polygons_gdf[polygons_gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

    if len(polygons_gdf) == 0:
        apartments_gdf[output_col] = 0.0
        print(f"{output_col}: в слое нет полигонов, записан 0.")
        return apartments_gdf

    buffers = apartments_gdf[["apt_uid", "geometry"]].copy()
    buffers["geometry"] = buffers.geometry.buffer(radius_m)

    joined = gpd.sjoin(
        buffers,
        polygons_gdf[["geometry"]],
        how="inner",
        predicate="intersects"
    )

    if len(joined) == 0:
        apartments_gdf[output_col] = 0.0
        print(f"{output_col}: пересечений нет, записан 0.")
        return apartments_gdf

    right_geom = (
        polygons_gdf[["geometry"]]
        .reset_index()
        .rename(columns={"index": "index_right", "geometry": "right_geometry"})
    )

    joined = joined.merge(
        right_geom,
        on="index_right",
        how="left"
    )

    joined["intersection_area"] = joined.apply(
        lambda row: row.geometry.intersection(row.right_geometry).area
        if row.right_geometry is not None else 0,
        axis=1
    )

    areas = (
        joined
        .groupby("apt_uid")["intersection_area"]
        .sum()
        .reset_index(name=output_col)
    )

    apartments_gdf = apartments_gdf.merge(
        areas,
        on="apt_uid",
        how="left"
    )

    apartments_gdf[output_col] = apartments_gdf[output_col].fillna(0.0)

    print(f"{output_col}: площадь посчитана в радиусе {radius_m} м.")
    return apartments_gdf

def standardize_social_layer(social_gdf):
    social_gdf = social_gdf.copy()

    possible_cols = [
        "subcategory",
        "amenity",
        "healthcare",
        "building",
        "name",
        "type"
    ]

    existing = [col for col in possible_cols if col in social_gdf.columns]

    if len(existing) == 0:
        raise ValueError(
            "В social-слое нет полей subcategory/amenity/healthcare/name/type, "
            "по которым можно определить школы, сады и медицину."
        )

    combined = pd.Series("", index=social_gdf.index, dtype="object")

    for col in existing:
        combined = combined + " " + social_gdf[col].astype(str).str.lower().fillna("")

    social_gdf["subcategory_std"] = np.nan

    school_mask = combined.str.contains(
        r"\bschool\b|школ|лицей|гимназ",
        regex=True,
        na=False
    )

    kindergarten_mask = combined.str.contains(
        r"\bkindergarten\b|детск|сад",
        regex=True,
        na=False
    )

    clinic_mask = combined.str.contains(
        r"\bclinic\b|\bhospital\b|\bdoctors\b|\bdentist\b|поликлин|больниц|медицин|clinic|hospital",
        regex=True,
        na=False
    )

    social_gdf.loc[school_mask, "subcategory_std"] = "school"
    social_gdf.loc[kindergarten_mask, "subcategory_std"] = "kindergarten"
    social_gdf.loc[clinic_mask, "subcategory_std"] = "clinic"

    social_gdf = social_gdf[social_gdf["subcategory_std"].notna()].copy()

    print("\nСоциальная инфраструктура после классификации:")
    print(social_gdf["subcategory_std"].value_counts(dropna=False))

    return social_gdf

def get_social_subset(social_gdf, category):
    return social_gdf[social_gdf["subcategory_std"] == category].copy()

apartments = read_gpkg_auto(
    APARTMENTS_PREPARED_GPKG,
    APARTMENTS_PREPARED_LAYER,
    label="очищенные квартиры"
)

apartments = prepare_layer(apartments, target_crs=TARGET_CRS)

required_apartment_cols = [
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "material_type",
    "geometry"
]

missing = [col for col in required_apartment_cols if col not in apartments.columns]

if missing:
    raise ValueError(f"В apartments_prepared не хватает колонок: {missing}")

apartments = apartments.reset_index(drop=True).copy()
apartments["apt_uid"] = np.arange(len(apartments))

base_cols = [
    "apt_uid",
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "building_year",
    "material_type",
    "geometry"
]

optional_cols = [
    "url",
    "address",
    "latitude",
    "longitude",
    "ln_price",
    "ln_total_area",
    "area_per_room",
    "floor_ratio",
    "is_first_floor",
    "is_last_floor",
    "is_middle_floor",
    "building_age",
    "building_age_sq",
    "rooms_group",
    "floor_type"
]

existing_optional = [col for col in optional_cols if col in apartments.columns]

apartments_base = apartments[
    [col for col in base_cols if col != "geometry"] + existing_optional + ["geometry"]
].copy()

if os.path.exists(APARTMENTS_MODEL_BASE_GPKG):
    os.remove(APARTMENTS_MODEL_BASE_GPKG)

save_layer(
    apartments_base,
    APARTMENTS_MODEL_BASE_GPKG,
    APARTMENTS_MODEL_BASE_LAYER
)

apartments_base.drop(columns="geometry").to_csv(
    "apartments_model_base.csv",
    index=False
)

print("\nБазовый слой квартир сохранен.")
print("Квартир:", len(apartments_base))

hexes = read_gpkg_auto(
    HEX_GPKG,
    HEX_LAYER,
    label="гексагоны с зонами"
)

hexes = prepare_layer(hexes, target_crs=TARGET_CRS)
hexes = harmonize_zone_layer(hexes)
apartments_base = drop_old_zone_columns(apartments_base)

required_hex_cols = [
    "hex_id",
    "zone_name",
    "zone_rank",
    "geometry"
]

missing_hex = [col for col in required_hex_cols if col not in hexes.columns]

if missing_hex:
    raise ValueError(f"В слое гексагонов не хватает колонок: {missing_hex}")

hexes_small = hexes[
    [
        "hex_id",
        "zone_name",
        "zone_rank",
        "geometry"
    ]
].copy()

hexes_small["zone_name"] = (
    hexes_small["zone_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("\nРаспределение гексагонов по зонам:")
print(hexes_small["zone_name"].value_counts(dropna=False))

apartments_zoned = gpd.sjoin(
    apartments_base,
    hexes_small,
    how="left",
    predicate="within"
)

if "index_right" in apartments_zoned.columns:
    apartments_zoned = apartments_zoned.drop(columns=["index_right"])

apartments_zoned["zone_core"] = np.where(
    apartments_zoned["zone_name"] == "core",
    1,
    0
)

apartments_zoned["zone_semi_periphery"] = np.where(
    apartments_zoned["zone_name"] == "semi_periphery",
    1,
    0
)

zone_missing = apartments_zoned["zone_name"].isna()

apartments_zoned.loc[
    zone_missing,
    ["zone_rank", "zone_core", "zone_semi_periphery"]
] = np.nan

print("\nРаспределение квартир по зонам до фильтра:")
print(apartments_zoned["zone_name"].value_counts(dropna=False))
print("Квартир без зоны:", apartments_zoned["zone_name"].isna().sum())
print("Доля без зоны:", round(apartments_zoned["zone_name"].isna().mean() * 100, 2), "%")

n_before_zone_filter = len(apartments_zoned)

apartments_zoned = apartments_zoned[
    apartments_zoned["zone_name"].isin(["periphery", "semi_periphery", "core"])
].copy()

n_after_zone_filter = len(apartments_zoned)

print("\nФильтр по зонам:")
print("Было:", n_before_zone_filter)
print("Осталось:", n_after_zone_filter)
print("Удалено:", n_before_zone_filter - n_after_zone_filter)

if os.path.exists(APARTMENTS_WITH_ZONES_GPKG):
    os.remove(APARTMENTS_WITH_ZONES_GPKG)

save_layer(
    apartments_zoned,
    APARTMENTS_WITH_ZONES_GPKG,
    APARTMENTS_WITH_ZONES_LAYER
)

apartments_zoned.drop(columns="geometry").to_csv(
    "apartments_with_zones.csv",
    index=False
)

print("\nСлой квартир с зонами сохранен.")

external = {}

for key, params in LAYER_PATHS.items():
    g = read_gpkg_auto(
        params["gpkg"],
        params["layer"],
        label=key
    )

    g = prepare_layer(g, target_crs=TARGET_CRS)

    g = g.reset_index(drop=True).copy()
    g[f"{key}_uid"] = np.arange(len(g))

    external[key] = g

print("\nВсе внешние слои загружены и приведены к CRS:", TARGET_CRS)

if SAVE_PROCESSED_LAYERS:
    if os.path.exists(PROCESSED_LAYERS_GPKG):
        os.remove(PROCESSED_LAYERS_GPKG)

    for i, (key, g) in enumerate(external.items()):
        save_layer(
            g,
            PROCESSED_LAYERS_GPKG,
            key,
            remove_file=False
        )

    print("\nОбработанные внешние слои сохранены в:", PROCESSED_LAYERS_GPKG)

social = standardize_social_layer(external["social"])

schools = get_social_subset(social, "school")
kindergartens = get_social_subset(social, "kindergarten")
clinics = get_social_subset(social, "clinic")

features = apartments_zoned.copy()

features = add_nearest_distance(
    features,
    external["metro_mcc"],
    "dist_nearest_metro_mcc"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_metro_mcc",
    1000,
    "metro_mcc_1000m"
)

features = add_nearest_distance(
    features,
    external["mcd"],
    "dist_nearest_mcd"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_mcd",
    1500,
    "mcd_1500m"
)

features = add_nearest_distance(
    features,
    external["stops"],
    "dist_nearest_stop"
)

features = add_points_count_within_radius(
    features,
    external["stops"],
    500,
    "stops_500m"
)

features = add_nearest_distance(
    features,
    schools,
    "dist_nearest_school"
)

features = add_points_count_within_radius(
    features,
    schools,
    1000,
    "schools_1000m"
)

features = add_nearest_distance(
    features,
    kindergartens,
    "dist_nearest_kindergarten"
)

features = add_points_count_within_radius(
    features,
    kindergartens,
    1000,
    "kindergartens_1000m"
)

features = add_nearest_distance(
    features,
    clinics,
    "dist_nearest_clinic"
)

features = add_points_count_within_radius(
    features,
    clinics,
    1000,
    "clinics_1000m"
)

features = add_nearest_distance(
    features,
    external["parks"],
    "dist_nearest_green"
)

features = add_polygon_area_within_radius(
    features,
    external["parks"],
    1000,
    "green_area_1000m"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_green",
    500,
    "green_500m"
)

features = add_nearest_distance(
    features,
    external["rivers"],
    "dist_nearest_water"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_water",
    500,
    "water_500m"
)

features = add_polygon_area_within_radius(
    features,
    external["rivers"],
    1000,
    "water_area_1000m"
)

features = add_nearest_distance(
    features,
    external["highway"],
    "dist_nearest_major_road"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_major_road",
    300,
    "major_road_300m"
)

features = add_nearest_distance(
    features,
    external["railway"],
    "dist_nearest_railway"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_railway",
    300,
    "railway_300m"
)

features = add_nearest_distance(
    features,
    external["industrial"],
    "dist_nearest_industrial"
)

features = add_polygon_area_within_radius(
    features,
    external["industrial"],
    1000,
    "industrial_area_1000m"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_industrial",
    500,
    "industrial_500m"
)

features = add_nearest_distance(
    features,
    external["aerodromes"],
    "dist_nearest_aerodrome"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_aerodrome",
    3000,
    "aerodrome_3000m"
)

features = add_nearest_distance(
    features,
    external["waste"],
    "dist_nearest_waste"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_waste",
    1000,
    "waste_1000m"
)

features = add_nearest_distance(
    features,
    external["cemeteries"],
    "dist_nearest_cemetery"
)

features = add_dummy_from_distance(
    features,
    "dist_nearest_cemetery",
    500,
    "cemetery_500m"
)

features = add_polygon_area_within_radius(
    features,
    external["cemeteries"],
    1000,
    "cemetery_area_1000m"
)

transport_cols = [
    "dist_nearest_metro_mcc",
    "metro_mcc_1000m",
    "dist_nearest_mcd",
    "mcd_1500m",
    "dist_nearest_stop",
    "stops_500m"
]

positive_cols = [
    "dist_nearest_school",
    "schools_1000m",
    "dist_nearest_kindergarten",
    "kindergartens_1000m",
    "dist_nearest_clinic",
    "clinics_1000m",
    "dist_nearest_green",
    "green_area_1000m",
    "green_500m",
    "dist_nearest_water",
    "water_500m",
    "water_area_1000m"
]

negative_cols = [
    "dist_nearest_major_road",
    "major_road_300m",
    "dist_nearest_railway",
    "railway_300m",
    "dist_nearest_industrial",
    "industrial_area_1000m",
    "industrial_500m",
    "dist_nearest_aerodrome",
    "aerodrome_3000m",
    "dist_nearest_waste",
    "waste_1000m",
    "dist_nearest_cemetery",
    "cemetery_500m",
    "cemetery_area_1000m"
]

all_new_cols = transport_cols + positive_cols + negative_cols

print("\n" + "=" * 70)
print("Проверка новых признаков:")
print(features[all_new_cols].describe().T)

print("\nСредние значения новых признаков по зонам:")
print(
    features
    .groupby("zone_name")[all_new_cols]
    .mean()
    .round(2)
)

main_cols = [
    "apt_uid",
    "id",
    "price",
    "price_m2",
    "ln_price_m2",
    "total_area",
    "rooms_count",
    "floor_number",
    "floors_count",
    "floor_ratio",
    "is_first_floor",
    "is_last_floor",
    "building_year",
    "building_age",
    "building_age_sq",
    "material_type",
    "hex_id",
    "zone_name",
    "zone_rank",
    "zone_core",
    "zone_semi_periphery"
]

existing_main_cols = [
    col for col in main_cols
    if col in features.columns
]

extra_cols = [
    col for col in features.columns
    if col not in existing_main_cols + ["geometry"]
]

features = features[
    existing_main_cols + extra_cols + ["geometry"]
].copy()

if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

save_layer(
    features,
    OUTPUT_GPKG,
    OUTPUT_LAYER
)

features.drop(columns="geometry").to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n" + "=" * 70)
print("ГОТОВО.")
print(f"Финальный GPKG: {OUTPUT_GPKG}, слой: {OUTPUT_LAYER}")
print(f"Финальный CSV: {OUTPUT_CSV}")
print("Количество квартир:", len(features))
print("CRS:", features.crs)



Загружен слой: очищенные квартиры
Файл: apartments_prepared.gpkg
Слой: apartments_prepared
Количество объектов: 31614
CRS: EPSG:32637
Типы геометрии:
Point    31614
Name: count, dtype: int64

Базовый слой квартир сохранен.
Квартир: 31614

Загружен слой: гексагоны с зонами
Файл: moscow_hex_zones_two_variants_activity_4blocks_FINAL-2.gpkg
Слой: zones_two_variants_final
Количество объектов: 1467
CRS: EPSG:32637
Типы геометрии:
MultiPolygon    1467
Name: count, dtype: int64

Распределение гексагонов после приведения нового зонирования:
zone_name
periphery         1018
semi_periphery     325
core                99
non_urban           54
Name: count, dtype: int64

Распределение гексагонов по зонам:
zone_name
periphery         1018
semi_periphery     325
core                99
non_urban           54
Name: count, dtype: int64

Распределение квартир по зонам до фильтра:
zone_name
periphery         13256
semi_periphery    12502
core               5812
NaN                  35
non_urban          

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

INPUT_GPKG = "apartments_hedonic_features.gpkg"
INPUT_LAYER = "apartments_hedonic_features"

OUTPUT_GPKG = "apartments_hedonic_indices.gpkg"
OUTPUT_LAYER = "apartments_hedonic_indices"
OUTPUT_CSV = "apartments_hedonic_indices.csv"

if not os.path.exists(INPUT_GPKG):
    raise FileNotFoundError(
        f"Не найден файл {INPUT_GPKG}."
    )

gdf = gpd.read_file(INPUT_GPKG, layer=INPUT_LAYER)

print("Квартир:", len(gdf))
print("CRS:", gdf.crs)
print("Колонки:")
print(gdf.columns.tolist())

def safe_log1p(series):
    s = pd.to_numeric(series, errors="coerce")
    s = s.where(s >= 0, np.nan)
    return np.log1p(s)

def zscore(series):
    s = pd.to_numeric(series, errors="coerce")
    mean = s.mean()
    std = s.std()

    if pd.isna(std) or std == 0:
        return pd.Series(0, index=s.index)

    return (s - mean) / std

def fill_median(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.fillna(s.median())

def row_mean(df, cols):
    return df[cols].mean(axis=1)

if "ln_total_area" not in gdf.columns:
    gdf["ln_total_area"] = np.log(gdf["total_area"])

gdf["idx_ln_total_area"] = fill_median(gdf["ln_total_area"])

gdf["idx_rooms_count"] = fill_median(gdf["rooms_count"])

gdf["floor_comfort_score"] = np.nan

gdf.loc[gdf["is_first_floor"] == 1, "floor_comfort_score"] = 0
gdf.loc[gdf["is_last_floor"] == 1, "floor_comfort_score"] = 0.5
gdf.loc[gdf["is_middle_floor"] == 1, "floor_comfort_score"] = 1

gdf["idx_floor_comfort_score"] = fill_median(gdf["floor_comfort_score"])

if "building_age" not in gdf.columns:
    CURRENT_YEAR = 2026
    gdf["building_age"] = CURRENT_YEAR - pd.to_numeric(gdf["building_year"], errors="coerce")

gdf["idx_building_newness"] = -safe_log1p(gdf["building_age"])
gdf["idx_building_newness"] = fill_median(gdf["idx_building_newness"])

def material_to_score(x):
    if pd.isna(x):
        return np.nan

    s = str(x).lower().strip()

    if any(v in s for v in ["монолит", "monolith", "monolithic"]):
        return 1.0

    if any(v in s for v in ["кирп", "brick"]):
        return 0.8

    if any(v in s for v in ["блок", "block"]):
        return 0.6

    if any(v in s for v in ["панел", "panel"]):
        return 0.4

    if any(v in s for v in ["дерев", "wood"]):
        return 0.3

    return 0.5

gdf["material_score"] = gdf["material_type"].apply(material_to_score)
gdf["idx_material_score"] = fill_median(gdf["material_score"])

internal_raw_cols = [
    "idx_ln_total_area",
    "idx_rooms_count",
    "idx_floor_comfort_score",
    "idx_building_newness",
    "idx_material_score"
]

internal_z_cols = []

for col in internal_raw_cols:
    z_col = "z_" + col
    gdf[z_col] = zscore(gdf[col])
    internal_z_cols.append(z_col)

gdf["internal_index"] = row_mean(gdf, internal_z_cols)

gdf["idx_schools_1000m"] = safe_log1p(gdf["schools_1000m"])
gdf["idx_kindergartens_1000m"] = safe_log1p(gdf["kindergartens_1000m"])
gdf["idx_clinics_1000m"] = safe_log1p(gdf["clinics_1000m"])

gdf["idx_green_area_1000m"] = safe_log1p(gdf["green_area_1000m"])

gdf["idx_green_proximity"] = -safe_log1p(gdf["dist_nearest_green"])
gdf["idx_water_proximity"] = -safe_log1p(gdf["dist_nearest_water"])

external_positive_raw_cols = [
    "idx_schools_1000m",
    "idx_kindergartens_1000m",
    "idx_clinics_1000m",
    "idx_green_area_1000m",
    "idx_green_proximity",
    "idx_water_proximity"
]

external_positive_z_cols = []

for col in external_positive_raw_cols:
    gdf[col] = fill_median(gdf[col])
    z_col = "z_" + col
    gdf[z_col] = zscore(gdf[col])
    external_positive_z_cols.append(z_col)

gdf["external_positive_index"] = row_mean(gdf, external_positive_z_cols)

gdf["idx_major_road_300m"] = pd.to_numeric(gdf["major_road_300m"], errors="coerce").fillna(0)
gdf["idx_railway_300m"] = pd.to_numeric(gdf["railway_300m"], errors="coerce").fillna(0)
gdf["idx_industrial_area_1000m"] = safe_log1p(gdf["industrial_area_1000m"])
gdf["idx_waste_1000m"] = pd.to_numeric(gdf["waste_1000m"], errors="coerce").fillna(0)

external_negative_raw_cols = [
    "idx_major_road_300m",
    "idx_railway_300m",
    "idx_industrial_area_1000m",
    "idx_waste_1000m"
]

external_negative_z_cols = []

for col in external_negative_raw_cols:
    gdf[col] = fill_median(gdf[col])
    z_col = "z_" + col
    gdf[z_col] = zscore(gdf[col])
    external_negative_z_cols.append(z_col)

gdf["external_negative_index"] = row_mean(gdf, external_negative_z_cols)

gdf["idx_metro_mcc_proximity"] = -safe_log1p(gdf["dist_nearest_metro_mcc"])
gdf["idx_mcd_proximity"] = -safe_log1p(gdf["dist_nearest_mcd"])
gdf["idx_stop_proximity"] = -safe_log1p(gdf["dist_nearest_stop"])
gdf["idx_stops_500m"] = safe_log1p(gdf["stops_500m"])

transport_raw_cols = [
    "idx_metro_mcc_proximity",
    "idx_mcd_proximity",
    "idx_stop_proximity",
    "idx_stops_500m"
]

transport_z_cols = []

for col in transport_raw_cols:
    gdf[col] = fill_median(gdf[col])
    z_col = "z_" + col
    gdf[z_col] = zscore(gdf[col])
    transport_z_cols.append(z_col)

gdf["transport_accessibility_index"] = row_mean(gdf, transport_z_cols)

index_cols = [
    "internal_index",
    "external_positive_index",
    "external_negative_index",
    "transport_accessibility_index"
]

print("\nОписание индексов:")
print(gdf[index_cols].describe().T.round(3))

print("\nСредние значения индексов по зонам:")
print(
    gdf
    .groupby("zone_name")[index_cols]
    .mean()
    .round(3)
)

print("\nМедианные значения индексов по зонам:")
print(
    gdf
    .groupby("zone_name")[index_cols]
    .median()
    .round(3)
)

print("\nКорреляции между индексами:")
print(
    gdf[index_cols + ["ln_price_m2", "zone_rank"]]
    .corr()
    .round(3)
)

if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

gdf.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG"
)

gdf.drop(columns="geometry").to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nГотово.")
print(f"GPKG сохранен: {OUTPUT_GPKG}, слой: {OUTPUT_LAYER}")
print(f"CSV сохранен: {OUTPUT_CSV}")
print("Количество квартир:", len(gdf))


Квартир: 31570
CRS: EPSG:32637
Колонки:
['apt_uid', 'id', 'price', 'price_m2', 'ln_price_m2', 'total_area', 'rooms_count', 'floor_number', 'floors_count', 'floor_ratio', 'is_first_floor', 'is_last_floor', 'building_year', 'building_age', 'building_age_sq', 'material_type', 'hex_id', 'zone_name', 'zone_rank', 'zone_core', 'zone_semi_periphery', 'url', 'address', 'latitude', 'longitude', 'ln_price', 'ln_total_area', 'area_per_room', 'is_middle_floor', 'rooms_group', 'floor_type', 'dist_nearest_metro_mcc', 'metro_mcc_1000m', 'dist_nearest_mcd', 'mcd_1500m', 'dist_nearest_stop', 'stops_500m', 'dist_nearest_school', 'schools_1000m', 'dist_nearest_kindergarten', 'kindergartens_1000m', 'dist_nearest_clinic', 'clinics_1000m', 'dist_nearest_green', 'green_area_1000m', 'green_500m', 'dist_nearest_water', 'water_500m', 'water_area_1000m', 'dist_nearest_major_road', 'major_road_300m', 'dist_nearest_railway', 'railway_300m', 'dist_nearest_industrial', 'industrial_area_1000m', 'industrial_500m', 'di

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

INPUT_GPKG = "apartments_hedonic_indices.gpkg"
INPUT_LAYER = "apartments_hedonic_indices"

DEPENDENT_VAR = "ln_price_m2"

FILTER_ROOMS_1_3 = True

TEST_SIZE = 0.2
RANDOM_STATE = 42

OUTPUT_RESULTS_CSV = f"model_6_spatial_nonlinear_controls_{DEPENDENT_VAR}_metrics.csv"
OUTPUT_COEF_CSV = f"model_6_spatial_nonlinear_controls_{DEPENDENT_VAR}_coefficients.csv"
OUTPUT_SUMMARY_TXT = f"model_6_spatial_nonlinear_controls_{DEPENDENT_VAR}_summary.txt"

if not os.path.exists(INPUT_GPKG):
    raise FileNotFoundError(
        f"Не найден файл {INPUT_GPKG}."
    )

gdf = gpd.read_file(INPUT_GPKG, layer=INPUT_LAYER)

print("Всего квартир:", len(gdf))
print("CRS:", gdf.crs)

gdf["price"] = pd.to_numeric(gdf["price"], errors="coerce")

if "ln_price" not in gdf.columns:
    gdf["ln_price"] = np.log(gdf["price"])

gdf["rooms_count_num"] = pd.to_numeric(gdf["rooms_count"], errors="coerce")

print("\nРаспределение rooms_count до фильтра:")
print(gdf["rooms_count_num"].value_counts(dropna=False).sort_index())

if FILTER_ROOMS_1_3:
    gdf = gdf[gdf["rooms_count_num"].isin([1, 2, 3])].copy()

print("\nКвартир после фильтра:", len(gdf))
print("\nРаспределение по зонам:")
print(gdf["zone_name"].value_counts(dropna=False))

def safe_log1p(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s.where(s >= 0, np.nan)
    return np.log1p(s)

def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    mean = s.mean()
    std = s.std()

    if pd.isna(std) or std == 0:
        return pd.Series(0, index=s.index)

    return (s - mean) / std

def material_to_score(x):
    if pd.isna(x):
        return np.nan

    s = str(x).lower().strip()

    if any(v in s for v in ["монолит", "monolith", "monolithic"]):
        return 1.0
    if any(v in s for v in ["кирп", "brick"]):
        return 0.8
    if any(v in s for v in ["блок", "block"]):
        return 0.6
    if any(v in s for v in ["панел", "panel"]):
        return 0.4
    if any(v in s for v in ["дерев", "wood"]):
        return 0.3

    return 0.5

df = gdf.copy()

df["total_area"] = pd.to_numeric(df["total_area"], errors="coerce")

if "ln_total_area" not in df.columns:
    df["ln_total_area"] = np.log(df["total_area"])

if "floor_comfort_score" not in df.columns:
    df["floor_comfort_score"] = np.nan
    df.loc[df["is_first_floor"] == 1, "floor_comfort_score"] = 0
    df.loc[df["is_last_floor"] == 1, "floor_comfort_score"] = 0.5
    df.loc[df["is_middle_floor"] == 1, "floor_comfort_score"] = 1

if "material_score" not in df.columns:
    df["material_score"] = df["material_type"].apply(material_to_score)

df["building_newness"] = -safe_log1p(df["building_age"])

df["metro_mcc_proximity"] = -safe_log1p(df["dist_nearest_metro_mcc"])
df["mcd_proximity"] = -safe_log1p(df["dist_nearest_mcd"])
df["stop_proximity"] = -safe_log1p(df["dist_nearest_stop"])
df["log_stops_500m"] = safe_log1p(df["stops_500m"])

df["log_schools_1000m"] = safe_log1p(df["schools_1000m"])
df["log_kindergartens_1000m"] = safe_log1p(df["kindergartens_1000m"])
df["log_clinics_1000m"] = safe_log1p(df["clinics_1000m"])
df["log_green_area_1000m"] = safe_log1p(df["green_area_1000m"])

df["green_proximity"] = -safe_log1p(df["dist_nearest_green"])
df["water_proximity"] = -safe_log1p(df["dist_nearest_water"])

df["log_industrial_area_1000m"] = safe_log1p(df["industrial_area_1000m"])

df["x"] = df.geometry.x
df["y"] = df.geometry.y

core_points = df[df["zone_name"] == "core"].copy()

core_x = core_points.geometry.x.mean()
core_y = core_points.geometry.y.mean()

df["dist_to_core_center"] = np.sqrt(
    (df["x"] - core_x) ** 2 +
    (df["y"] - core_y) ** 2
)

df["log_dist_to_core_center"] = np.log1p(df["dist_to_core_center"])

df["x_centered"] = df["x"] - df["x"].mean()
df["y_centered"] = df["y"] - df["y"].mean()

df["x2"] = df["x_centered"] ** 2
df["y2"] = df["y_centered"] ** 2
df["xy"] = df["x_centered"] * df["y_centered"]

component_raw_cols = [

    "ln_total_area",
    "rooms_count",
    "floor_comfort_score",
    "building_newness",
    "material_score",

    "metro_mcc_proximity",
    "mcd_proximity",
    "stop_proximity",
    "log_stops_500m",

    "log_schools_1000m",
    "log_kindergartens_1000m",
    "log_clinics_1000m",
    "log_green_area_1000m",
    "green_proximity",
    "water_proximity",

    "major_road_300m",
    "railway_300m",
    "log_industrial_area_1000m",
    "waste_1000m"
]

for col in component_raw_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())
    df["z_" + col] = zscore(df[col])

spatial_raw_cols = [
    "x_centered",
    "y_centered",
    "x2",
    "y2",
    "xy",
    "log_dist_to_core_center"
]

for col in spatial_raw_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())
    df["z_" + col] = zscore(df[col])

zone_vars = [
    "zone_core",
    "zone_semi_periphery"
]

component_vars = [

    "z_ln_total_area",
    "z_rooms_count",
    "z_floor_comfort_score",
    "z_building_newness",
    "z_material_score",

    "z_metro_mcc_proximity",
    "z_mcd_proximity",
    "z_stop_proximity",
    "z_log_stops_500m",

    "z_log_schools_1000m",
    "z_log_kindergartens_1000m",
    "z_log_clinics_1000m",
    "z_log_green_area_1000m",
    "z_green_proximity",
    "z_water_proximity",

    "z_major_road_300m",
    "z_railway_300m",
    "z_log_industrial_area_1000m",
    "z_waste_1000m"
]

interaction_vars = []

for component in component_vars:
    for zone in zone_vars:
        new_col = f"{component}_x_{zone}"
        df[new_col] = df[component] * df[zone]
        interaction_vars.append(new_col)

spatial_vars = [
    "z_x_centered",
    "z_y_centered",
    "z_x2",
    "z_y2",
    "z_xy",
    "z_log_dist_to_core_center"
]

MODEL_VARS = zone_vars + component_vars + interaction_vars + spatial_vars

print("\nКоличество признаков model_6:", len(MODEL_VARS))

model_df = df[[DEPENDENT_VAR, "zone_name"] + MODEL_VARS].copy()

for col in [DEPENDENT_VAR] + MODEL_VARS:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

for col in MODEL_VARS:
    model_df[col] = model_df[col].fillna(model_df[col].median())

n_before = len(model_df)
model_df = model_df.dropna(subset=[DEPENDENT_VAR, "zone_name"]).copy()
n_after = len(model_df)

print("\nНаблюдений до удаления NaN:", n_before)
print("Наблюдений в модели:", n_after)
print("Удалено:", n_before - n_after)

train_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=model_df["zone_name"]
)

print("\nTrain:", len(train_df))
print("Test:", len(test_df))

y_train = train_df[DEPENDENT_VAR]
X_train = sm.add_constant(train_df[MODEL_VARS], has_constant="add")

y_test = test_df[DEPENDENT_VAR]
X_test = sm.add_constant(test_df[MODEL_VARS], has_constant="add")

model = sm.OLS(y_train, X_train).fit(cov_type="HC3")

print(model.summary())

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

metrics = {
    "model": "model_6_spatial_nonlinear_controls",
    "dependent_var": DEPENDENT_VAR,
    "rooms_filter": "1_3_only" if FILTER_ROOMS_1_3 else "all",
    "n_total": len(model_df),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "n_features": len(MODEL_VARS),
    "r2_train": model.rsquared,
    "adj_r2_train": model.rsquared_adj,
    "r2_test": r2_score(y_test, y_test_pred),
    "rmse_train_log": np.sqrt(mean_squared_error(y_train, y_train_pred)),
    "rmse_test_log": np.sqrt(mean_squared_error(y_test, y_test_pred)),
    "mae_train_log": mean_absolute_error(y_train, y_train_pred),
    "mae_test_log": mean_absolute_error(y_test, y_test_pred),
    "aic_train": model.aic,
    "bic_train": model.bic
}

metrics_df = pd.DataFrame([metrics])

metrics_df_rounded = metrics_df.copy()

for col in [
    "r2_train",
    "adj_r2_train",
    "r2_test",
    "rmse_train_log",
    "rmse_test_log",
    "mae_train_log",
    "mae_test_log",
    "aic_train",
    "bic_train"
]:
    metrics_df_rounded[col] = metrics_df_rounded[col].round(4)

print("\nМетрики model_6:")
display(metrics_df_rounded)

metrics_df_rounded.to_csv(OUTPUT_RESULTS_CSV, index=False)

coef_table = pd.DataFrame({
    "variable": model.params.index,
    "coef_log": model.params.values,
    "std_err": model.bse.values,
    "t_value": model.tvalues.values,
    "p_value": model.pvalues.values,
    "ci_low": model.conf_int()[0].values,
    "ci_high": model.conf_int()[1].values
})

coef_table["effect_percent"] = (np.exp(coef_table["coef_log"]) - 1) * 100
coef_table["ci_low_percent"] = (np.exp(coef_table["ci_low"]) - 1) * 100
coef_table["ci_high_percent"] = (np.exp(coef_table["ci_high"]) - 1) * 100

coef_table_rounded = coef_table.round({
    "coef_log": 4,
    "std_err": 4,
    "t_value": 3,
    "p_value": 4,
    "ci_low": 4,
    "ci_high": 4,
    "effect_percent": 2,
    "ci_low_percent": 2,
    "ci_high_percent": 2
})

print("\nКоэффициенты model_6:")
display(coef_table_rounded)

coef_table_rounded.to_csv(OUTPUT_COEF_CSV, index=False)

with open(OUTPUT_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(str(model.summary()))
    f.write("\n\nМетрики:\n")
    for key, value in metrics.items():
        f.write(f"{key}: {value}\n")

print("\nГотово.")
print("Метрики сохранены:", OUTPUT_RESULTS_CSV)
print("Коэффициенты сохранены:", OUTPUT_COEF_CSV)
print("Summary сохранен:", OUTPUT_SUMMARY_TXT)


Всего квартир: 31570
CRS: EPSG:32637

Распределение rooms_count до фильтра:
rooms_count_num
1.0    5858
2.0    9885
3.0    8904
4.0    3315
5.0     991
6.0     295
NaN    2322
Name: count, dtype: int64

Квартир после фильтра: 24647

Распределение по зонам:
zone_name
periphery         11205
semi_periphery     9747
core               3695
Name: count, dtype: int64

Количество признаков model_6: 65

Наблюдений до удаления NaN: 24647
Наблюдений в модели: 24647
Удалено: 0

Train: 19717
Test: 4930
                            OLS Regression Results                            
Dep. Variable:            ln_price_m2   R-squared:                       0.621
Model:                            OLS   Adj. R-squared:                  0.620
Method:                 Least Squares   F-statistic:                     478.6
Date:                Sun, 07 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:02:41   Log-Likelihood:                -1991.0
No. Observations:            

,model,dependent_var,rooms_filter,n_total,n_train,n_test,n_features,r2_train,adj_r2_train,r2_test,rmse_train_log,rmse_test_log,mae_train_log,mae_test_log,aic_train,bic_train
0,model_6_spatial_nonlinear_controls,ln_price_m2,1_3_only,24647,19717,4930,65,0.6215,0.6202,0.607,0.2677,0.2812,0.1846,0.1916,4114.0994,4634.789



Коэффициенты model_6:


,variable,coef_log,std_err,t_value,p_value,ci_low,ci_high,effect_percent,ci_low_percent,ci_high_percent
0,const,12.9390,0.0064,2025.354,0.0000,12.9264,12.9515,41621686.30,41103779.93,42146118.25
1,zone_core,0.0434,0.0205,2.120,0.0340,0.0033,0.0835,4.43,0.33,8.71
2,zone_semi_periphery,-0.0022,0.0076,-0.283,0.7772,-0.0171,0.0128,-0.21,-1.69,1.28
3,z_ln_total_area,0.0025,0.0064,0.394,0.6933,-0.0101,0.0151,0.25,-1.00,1.52
4,z_rooms_count,-0.0600,0.0052,-11.527,0.0000,-0.0702,-0.0498,-5.82,-6.78,-4.86
...,...,...,...,...,...,...,...,...,...,...
61,z_y_centered,-0.0211,0.0020,-10.410,0.0000,-0.0251,-0.0172,-2.09,-2.48,-1.70
62,z_x2,-0.0441,0.0034,-12.925,0.0000,-0.0508,-0.0374,-4.31,-4.95,-3.67
63,z_y2,-0.0682,0.0041,-16.783,0.0000,-0.0762,-0.0603,-6.60,-7.34,-5.85
64,z_xy,-0.0035,0.0018,-1.942,0.0521,-0.0070,0.0000,-0.35,-0.70,0.00



Готово.
Метрики сохранены: model_6_spatial_nonlinear_controls_ln_price_m2_metrics.csv
Коэффициенты сохранены: model_6_spatial_nonlinear_controls_ln_price_m2_coefficients.csv
Summary сохранен: model_6_spatial_nonlinear_controls_ln_price_m2_summary.txt


In [ ]:
df["zone_core"] = (df["zone_name"] == "core").astype(int)
df["zone_semi_periphery"] = (df["zone_name"] == "semi_periphery").astype(int)

for component in component_vars:
    for zone in zone_vars:
        new_col = f"{component}_x_{zone}"
        df[new_col] = df[component] * df[zone]

model_df_full = df[[DEPENDENT_VAR, "zone_name"] + MODEL_VARS].copy()

for col in [DEPENDENT_VAR] + MODEL_VARS:
    model_df_full[col] = pd.to_numeric(model_df_full[col], errors="coerce")

for col in MODEL_VARS:
    model_df_full[col] = model_df_full[col].fillna(model_df_full[col].median())

model_df_full = model_df_full.dropna(subset=[DEPENDENT_VAR, "zone_name"]).copy()

y_full = model_df_full[DEPENDENT_VAR]
X_full = sm.add_constant(model_df_full[MODEL_VARS], has_constant="add")

model_full = sm.OLS(y_full, X_full).fit(cov_type="HC3")

print(model_full.summary())

coef_full = pd.DataFrame({
    "variable": model_full.params.index,
    "coef_log": model_full.params.values,
    "std_err": model_full.bse.values,
    "t_value": model_full.tvalues.values,
    "p_value": model_full.pvalues.values,
    "ci_low": model_full.conf_int()[0].values,
    "ci_high": model_full.conf_int()[1].values
})

coef_full["effect_percent"] = (np.exp(coef_full["coef_log"]) - 1) * 100
coef_full["ci_low_percent"] = (np.exp(coef_full["ci_low"]) - 1) * 100
coef_full["ci_high_percent"] = (np.exp(coef_full["ci_high"]) - 1) * 100

coef_full.to_csv(
    "model_6_full_sample_coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)

model_df_full["y_pred_full"] = model_full.predict(X_full)
model_df_full["residual_full"] = y_full - model_df_full["y_pred_full"]

model_df_full[[
    DEPENDENT_VAR,
    "y_pred_full",
    "residual_full",
    "zone_name"
]].to_csv(
    "model_6_full_sample_predictions_residuals.csv",
    index=False,
    encoding="utf-8-sig"
)

model = model_full
X = X_full.copy()
df_model = model_df_full.copy()

print("Финальная модель на всей выборке обучена.")
print("N:", len(model_df_full))
print("Коэффициенты сохранены: model_6_full_sample_coefficients.csv")
print("Предсказания и остатки сохранены: model_6_full_sample_predictions_residuals.csv")


                            OLS Regression Results                            
Dep. Variable:            ln_price_m2   R-squared:                       0.619
Model:                            OLS   Adj. R-squared:                  0.618
Method:                 Least Squares   F-statistic:                     597.3
Date:                Sun, 07 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:02:43   Log-Likelihood:                -2732.3
No. Observations:               24647   AIC:                             5597.
Df Residuals:                   24581   BIC:                             6132.
Df Model:                          65                                         
Covariance Type:                  HC3                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

INPUT_GPKG = "apartments_hedonic_indices.gpkg"
INPUT_LAYER = "apartments_hedonic_indices"

DEPENDENT_VAR = "ln_price_m2"

FILTER_ROOMS_1_3 = True

TEST_SIZE = 0.2
RANDOM_STATE = 42

N_SPLITS = 5

OUTPUT_DIR = "model_6_diagnostics_new_zones"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(INPUT_GPKG):
    raise FileNotFoundError(
        f"Не найден файл {INPUT_GPKG}."
    )

gdf = gpd.read_file(INPUT_GPKG, layer=INPUT_LAYER)

print("Файл:", INPUT_GPKG)
print("Слой:", INPUT_LAYER)
print("Всего квартир:", len(gdf))
print("CRS:", gdf.crs)

gdf["rooms_count_num"] = pd.to_numeric(gdf["rooms_count"], errors="coerce")

print("\nРаспределение rooms_count до фильтра:")
print(gdf["rooms_count_num"].value_counts(dropna=False).sort_index())

if FILTER_ROOMS_1_3:
    gdf = gdf[gdf["rooms_count_num"].isin([1, 2, 3])].copy()

print("\nКвартир после фильтра 1–3 комнаты:", len(gdf))
print("\nРаспределение по зонам:")
print(gdf["zone_name"].value_counts(dropna=False))

def safe_log1p(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s.where(s >= 0, np.nan)
    return np.log1p(s)

def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    mean = s.mean()
    std = s.std()

    if pd.isna(std) or std == 0:
        return pd.Series(0, index=s.index)

    return (s - mean) / std

def material_to_score(x):
    if pd.isna(x):
        return np.nan

    s = str(x).lower().strip()

    if any(v in s for v in ["монолит", "monolith", "monolithic"]):
        return 1.0
    if any(v in s for v in ["кирп", "brick"]):
        return 0.8
    if any(v in s for v in ["блок", "block"]):
        return 0.6
    if any(v in s for v in ["панел", "panel"]):
        return 0.4
    if any(v in s for v in ["дерев", "wood"]):
        return 0.3

    return 0.5

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def pseudo_r2(y_true, y_pred, baseline_mean=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if baseline_mean is None:
        baseline_mean = np.mean(y_true)

    sse_model = np.sum((y_true - y_pred) ** 2)
    sse_baseline = np.sum((y_true - baseline_mean) ** 2)

    if sse_baseline == 0:
        return np.nan

    return 1 - sse_model / sse_baseline

def fit_statsmodels_ols(train_df, y_col, x_cols):
    y_train = train_df[y_col]
    X_train = sm.add_constant(train_df[x_cols], has_constant="add")
    model = sm.OLS(y_train, X_train).fit(cov_type="HC3")
    return model

df = gdf.copy()

df["total_area"] = pd.to_numeric(df["total_area"], errors="coerce")

if "ln_total_area" not in df.columns:
    df["ln_total_area"] = np.log(df["total_area"])

if "floor_comfort_score" not in df.columns:
    df["floor_comfort_score"] = np.nan
    df.loc[df["is_first_floor"] == 1, "floor_comfort_score"] = 0
    df.loc[df["is_last_floor"] == 1, "floor_comfort_score"] = 0.5
    df.loc[df["is_middle_floor"] == 1, "floor_comfort_score"] = 1

if "material_score" not in df.columns:
    df["material_score"] = df["material_type"].apply(material_to_score)

df["building_newness"] = -safe_log1p(df["building_age"])

df["metro_mcc_proximity"] = -safe_log1p(df["dist_nearest_metro_mcc"])
df["mcd_proximity"] = -safe_log1p(df["dist_nearest_mcd"])
df["stop_proximity"] = -safe_log1p(df["dist_nearest_stop"])
df["log_stops_500m"] = safe_log1p(df["stops_500m"])

df["log_schools_1000m"] = safe_log1p(df["schools_1000m"])
df["log_kindergartens_1000m"] = safe_log1p(df["kindergartens_1000m"])
df["log_clinics_1000m"] = safe_log1p(df["clinics_1000m"])
df["log_green_area_1000m"] = safe_log1p(df["green_area_1000m"])

df["green_proximity"] = -safe_log1p(df["dist_nearest_green"])
df["water_proximity"] = -safe_log1p(df["dist_nearest_water"])

df["log_industrial_area_1000m"] = safe_log1p(df["industrial_area_1000m"])

df["x"] = df.geometry.x
df["y"] = df.geometry.y

core_points = df[df["zone_name"] == "core"].copy()

core_x = core_points.geometry.x.mean()
core_y = core_points.geometry.y.mean()

df["dist_to_core_center"] = np.sqrt(
    (df["x"] - core_x) ** 2 +
    (df["y"] - core_y) ** 2
)

df["log_dist_to_core_center"] = np.log1p(df["dist_to_core_center"])

df["x_centered"] = df["x"] - df["x"].mean()
df["y_centered"] = df["y"] - df["y"].mean()

df["x2"] = df["x_centered"] ** 2
df["y2"] = df["y_centered"] ** 2
df["xy"] = df["x_centered"] * df["y_centered"]

component_raw_cols = [

    "ln_total_area",
    "rooms_count",
    "floor_comfort_score",
    "building_newness",
    "material_score",

    "metro_mcc_proximity",
    "mcd_proximity",
    "stop_proximity",
    "log_stops_500m",

    "log_schools_1000m",
    "log_kindergartens_1000m",
    "log_clinics_1000m",
    "log_green_area_1000m",
    "green_proximity",
    "water_proximity",

    "major_road_300m",
    "railway_300m",
    "log_industrial_area_1000m",
    "waste_1000m"
]

for col in component_raw_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())
    df["z_" + col] = zscore(df[col])

spatial_raw_cols = [
    "x_centered",
    "y_centered",
    "x2",
    "y2",
    "xy",
    "log_dist_to_core_center"
]

for col in spatial_raw_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())
    df["z_" + col] = zscore(df[col])

zone_vars = [
    "zone_core",
    "zone_semi_periphery"
]

component_vars = [

    "z_ln_total_area",
    "z_rooms_count",
    "z_floor_comfort_score",
    "z_building_newness",
    "z_material_score",

    "z_metro_mcc_proximity",
    "z_mcd_proximity",
    "z_stop_proximity",
    "z_log_stops_500m",

    "z_log_schools_1000m",
    "z_log_kindergartens_1000m",
    "z_log_clinics_1000m",
    "z_log_green_area_1000m",
    "z_green_proximity",
    "z_water_proximity",

    "z_major_road_300m",
    "z_railway_300m",
    "z_log_industrial_area_1000m",
    "z_waste_1000m"
]

interaction_vars = []

for component in component_vars:
    for zone in zone_vars:
        new_col = f"{component}_x_{zone}"
        df[new_col] = df[component] * df[zone]
        interaction_vars.append(new_col)

spatial_vars = [
    "z_x_centered",
    "z_y_centered",
    "z_x2",
    "z_y2",
    "z_xy",
    "z_log_dist_to_core_center"
]

MODEL_VARS = zone_vars + component_vars + interaction_vars + spatial_vars

print("\nКоличество признаков MODEL 6:", len(MODEL_VARS))

if "hex_group_id" in df.columns:
    GROUP_COL = "hex_group_id"
elif "hex_id" in df.columns:
    GROUP_COL = "hex_id"
else:
    GROUP_COL = None

if GROUP_COL is None:
    print("\n нет hex_group_id/hex_id.")
    df["spatial_group"] = "row_" + df.index.astype(str)
else:
    df["spatial_group"] = df[GROUP_COL].astype("object")

    missing_groups = df["spatial_group"].isna().sum()
    print(f"\nГрупповая колонка для spatial CV: {GROUP_COL}")
    print("Пропусков в группах до заполнения:", missing_groups)

    df.loc[df["spatial_group"].isna(), "spatial_group"] = (
        "missing_group_" + df.loc[df["spatial_group"].isna()].index.astype(str)
    )

df["spatial_group"] = df["spatial_group"].astype(str)

print("Уникальных spatial_group:", df["spatial_group"].nunique())

model_df = df[[DEPENDENT_VAR, "zone_name", "spatial_group", "geometry"] + MODEL_VARS].copy()

for col in [DEPENDENT_VAR] + MODEL_VARS:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

for col in MODEL_VARS:
    model_df[col] = model_df[col].fillna(model_df[col].median())

n_before = len(model_df)
model_df = model_df.dropna(subset=[DEPENDENT_VAR, "zone_name", "spatial_group"]).copy()
n_after = len(model_df)

print("\nНаблюдений до удаления NaN:", n_before)
print("Наблюдений в модели:", n_after)
print("Удалено:", n_before - n_after)

train_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=model_df["zone_name"]
)

model_random = fit_statsmodels_ols(train_df, DEPENDENT_VAR, MODEL_VARS)

X_train = sm.add_constant(train_df[MODEL_VARS], has_constant="add")
X_test = sm.add_constant(test_df[MODEL_VARS], has_constant="add")

y_train = train_df[DEPENDENT_VAR]
y_test = test_df[DEPENDENT_VAR]

y_train_pred = model_random.predict(X_train)
y_test_pred = model_random.predict(X_test)

train_mean = y_train.mean()

random_metrics = {
    "validation": "random_train_test",
    "n_train": len(train_df),
    "n_test": len(test_df),

    "ols_r2_train": model_random.rsquared,
    "ols_adj_r2_train": model_random.rsquared_adj,

    "pseudo_r2_train": pseudo_r2(y_train, y_train_pred, baseline_mean=train_mean),
    "pseudo_r2_test": pseudo_r2(y_test, y_test_pred, baseline_mean=train_mean),

    "r2_test_sklearn": r2_score(y_test, y_test_pred),

    "rmse_train_log": rmse(y_train, y_train_pred),
    "rmse_test_log": rmse(y_test, y_test_pred),

    "mae_train_log": mean_absolute_error(y_train, y_train_pred),
    "mae_test_log": mean_absolute_error(y_test, y_test_pred)
}

random_metrics_df = pd.DataFrame([random_metrics]).round(4)

print("\nRandom train/test pseudo R² и метрики:")
display(random_metrics_df)

random_metrics_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_6_random_pseudo_r2_metrics.csv"),
    index=False
)

groups = model_df["spatial_group"].astype(str).values

group_cv = GroupKFold(n_splits=N_SPLITS)

fold_rows = []

all_y_true = []
all_y_pred = []
all_baseline_means = []

for fold, (train_idx, test_idx) in enumerate(
    group_cv.split(model_df, groups=groups),
    start=1
):
    fold_train = model_df.iloc[train_idx].copy()
    fold_test = model_df.iloc[test_idx].copy()

    fold_model = fit_statsmodels_ols(fold_train, DEPENDENT_VAR, MODEL_VARS)

    X_fold_train = sm.add_constant(fold_train[MODEL_VARS], has_constant="add")
    X_fold_test = sm.add_constant(fold_test[MODEL_VARS], has_constant="add")

    y_fold_train = fold_train[DEPENDENT_VAR]
    y_fold_test = fold_test[DEPENDENT_VAR]

    y_fold_train_pred = fold_model.predict(X_fold_train)
    y_fold_test_pred = fold_model.predict(X_fold_test)

    fold_train_mean = y_fold_train.mean()

    fold_row = {
        "fold": fold,
        "n_train": len(fold_train),
        "n_test": len(fold_test),
        "n_train_groups": fold_train["spatial_group"].nunique(),
        "n_test_groups": fold_test["spatial_group"].nunique(),

        "r2_train": fold_model.rsquared,
        "adj_r2_train": fold_model.rsquared_adj,

        "pseudo_r2_train": pseudo_r2(
            y_fold_train,
            y_fold_train_pred,
            baseline_mean=fold_train_mean
        ),
        "pseudo_r2_test": pseudo_r2(
            y_fold_test,
            y_fold_test_pred,
            baseline_mean=fold_train_mean
        ),

        "r2_test_sklearn": r2_score(y_fold_test, y_fold_test_pred),

        "rmse_train_log": rmse(y_fold_train, y_fold_train_pred),
        "rmse_test_log": rmse(y_fold_test, y_fold_test_pred),

        "mae_train_log": mean_absolute_error(y_fold_train, y_fold_train_pred),
        "mae_test_log": mean_absolute_error(y_fold_test, y_fold_test_pred)
    }

    fold_rows.append(fold_row)

    all_y_true.extend(y_fold_test.values)
    all_y_pred.extend(y_fold_test_pred.values)
    all_baseline_means.extend([fold_train_mean] * len(fold_test))

spatial_cv_folds = pd.DataFrame(fold_rows)

spatial_cv_summary = pd.DataFrame({
    "metric": [
        "pseudo_r2_test",
        "r2_test_sklearn",
        "rmse_test_log",
        "mae_test_log",
        "pseudo_r2_train",
        "rmse_train_log",
        "mae_train_log"
    ],
    "mean": [
        spatial_cv_folds["pseudo_r2_test"].mean(),
        spatial_cv_folds["r2_test_sklearn"].mean(),
        spatial_cv_folds["rmse_test_log"].mean(),
        spatial_cv_folds["mae_test_log"].mean(),
        spatial_cv_folds["pseudo_r2_train"].mean(),
        spatial_cv_folds["rmse_train_log"].mean(),
        spatial_cv_folds["mae_train_log"].mean()
    ],
    "std": [
        spatial_cv_folds["pseudo_r2_test"].std(),
        spatial_cv_folds["r2_test_sklearn"].std(),
        spatial_cv_folds["rmse_test_log"].std(),
        spatial_cv_folds["mae_test_log"].std(),
        spatial_cv_folds["pseudo_r2_train"].std(),
        spatial_cv_folds["rmse_train_log"].std(),
        spatial_cv_folds["mae_train_log"].std()
    ]
})

all_y_true = np.asarray(all_y_true)
all_y_pred = np.asarray(all_y_pred)
all_baseline_means = np.asarray(all_baseline_means)

sse_model_oof = np.sum((all_y_true - all_y_pred) ** 2)
sse_baseline_oof = np.sum((all_y_true - all_baseline_means) ** 2)

spatial_pseudo_r2_oof = 1 - sse_model_oof / sse_baseline_oof

spatial_oof_metrics = pd.DataFrame([{
    "validation": "spatial_GroupKFold_oof",
    "n_observations_oof": len(all_y_true),
    "n_splits": N_SPLITS,
    "spatial_pseudo_r2_oof": spatial_pseudo_r2_oof,
    "rmse_oof_log": rmse(all_y_true, all_y_pred),
    "mae_oof_log": mean_absolute_error(all_y_true, all_y_pred)
}])

print("\nSpatial CV по fold:")
display(spatial_cv_folds.round(4))

print("\nSpatial CV summary:")
display(spatial_cv_summary.round(4))

print("\nSpatial pseudo R² out-of-fold:")
display(spatial_oof_metrics.round(4))

spatial_cv_folds.round(4).to_csv(
    os.path.join(OUTPUT_DIR, "model_6_spatial_cv_folds.csv"),
    index=False
)

spatial_cv_summary.round(4).to_csv(
    os.path.join(OUTPUT_DIR, "model_6_spatial_cv_summary.csv"),
    index=False
)

spatial_oof_metrics.round(4).to_csv(
    os.path.join(OUTPUT_DIR, "model_6_spatial_pseudo_r2_oof.csv"),
    index=False
)

full_model = fit_statsmodels_ols(model_df, DEPENDENT_VAR, MODEL_VARS)

X_full = sm.add_constant(model_df[MODEL_VARS], has_constant="add")
y_full = model_df[DEPENDENT_VAR]

model_df["y_pred_full"] = full_model.predict(X_full)
model_df["residual_full"] = y_full - model_df["y_pred_full"]
model_df["abs_residual_full"] = model_df["residual_full"].abs()

full_metrics = {
    "validation": "full_sample",
    "n": len(model_df),
    "ols_r2": full_model.rsquared,
    "ols_adj_r2": full_model.rsquared_adj,
    "pseudo_r2_full": pseudo_r2(
        y_full,
        model_df["y_pred_full"],
        baseline_mean=y_full.mean()
    ),
    "rmse_full_log": rmse(y_full, model_df["y_pred_full"]),
    "mae_full_log": mean_absolute_error(y_full, model_df["y_pred_full"])
}

full_metrics_df = pd.DataFrame([full_metrics]).round(4)

print("\nFull sample metrics:")
display(full_metrics_df)

full_metrics_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_6_full_sample_metrics.csv"),
    index=False
)

try:
    from libpysal.weights import KNN
    from esda.moran import Moran

    residual_gdf = gpd.GeoDataFrame(
        model_df.copy(),
        geometry="geometry",
        crs=gdf.crs
    )

    residual_gdf = residual_gdf.dropna(subset=["residual_full"]).copy()

    K_NEIGHBORS = 8

    w_knn = KNN.from_dataframe(
        residual_gdf,
        k=K_NEIGHBORS
    )

    w_knn.transform = "R"

    moran = Moran(
        residual_gdf["residual_full"].values,
        w_knn,
        permutations=999
    )

    moran_results = pd.DataFrame([{
        "variable": "residual_full",
        "weights": f"KNN_k{K_NEIGHBORS}",
        "n": len(residual_gdf),
        "moran_I": moran.I,
        "expected_I": moran.EI,
        "z_norm": moran.z_norm,
        "p_norm": moran.p_norm,
        "z_rand": moran.z_rand,
        "p_rand": moran.p_rand,
        "p_sim": moran.p_sim,
        "permutations": 999
    }])

    print("\nMoran's I по остаткам полной модели:")
    display(moran_results.round(4))

    moran_results.round(4).to_csv(
        os.path.join(OUTPUT_DIR, "model_6_morans_i_residuals.csv"),
        index=False
    )

except ImportError:
    print("\nНе установлены libpysal/esda.")
    print("Запусти в отдельной ячейке:")
    print("!pip install libpysal esda")
    print("После установки перезапусти блок Moran's I.")

residuals_out = model_df[
    [
        DEPENDENT_VAR,
        "y_pred_full",
        "residual_full",
        "abs_residual_full",
        "zone_name",
        "spatial_group",
        "geometry"
    ]
].copy()

residuals_gdf = gpd.GeoDataFrame(
    residuals_out,
    geometry="geometry",
    crs=gdf.crs
)

residuals_gpkg = os.path.join(
    OUTPUT_DIR,
    "model_6_residuals.gpkg"
)

if os.path.exists(residuals_gpkg):
    os.remove(residuals_gpkg)

residuals_gdf.to_file(
    residuals_gpkg,
    layer="model_6_residuals",
    driver="GPKG"
)

residuals_gdf.drop(columns="geometry").to_csv(
    os.path.join(OUTPUT_DIR, "model_6_residuals.csv"),
    index=False
)

print("\nГотово.")
print("Все результаты сохранены в папку:", OUTPUT_DIR)


Файл: apartments_hedonic_indices.gpkg
Слой: apartments_hedonic_indices
Всего квартир: 31570
CRS: EPSG:32637

Распределение rooms_count до фильтра:
rooms_count_num
1.0    5858
2.0    9885
3.0    8904
4.0    3315
5.0     991
6.0     295
NaN    2322
Name: count, dtype: int64

Квартир после фильтра 1–3 комнаты: 24647

Распределение по зонам:
zone_name
periphery         11205
semi_periphery     9747
core               3695
Name: count, dtype: int64

Количество признаков MODEL 6: 65

Групповая колонка для spatial CV: hex_id
Пропусков в группах до заполнения: 0
Уникальных spatial_group: 975

Наблюдений до удаления NaN: 24647
Наблюдений в модели: 24647
Удалено: 0

Random train/test pseudo R² и метрики:


,validation,n_train,n_test,ols_r2_train,ols_adj_r2_train,pseudo_r2_train,pseudo_r2_test,r2_test_sklearn,rmse_train_log,rmse_test_log,mae_train_log,mae_test_log
0,random_train_test,19717,4930,0.6215,0.6202,0.6215,0.6072,0.607,0.2677,0.2812,0.1846,0.1916



Spatial CV по fold:


,fold,n_train,n_test,n_train_groups,n_test_groups,r2_train,adj_r2_train,pseudo_r2_train,pseudo_r2_test,r2_test_sklearn,rmse_train_log,rmse_test_log,mae_train_log,mae_test_log
0,1,19717,4930,781,194,0.6192,0.6179,0.6192,0.6119,0.6116,0.2701,0.2733,0.1860,0.1863
1,2,19717,4930,779,196,0.6168,0.6155,0.6168,0.6195,0.6195,0.2719,0.2666,0.1861,0.1876
2,3,19718,4929,780,195,0.6253,0.6241,0.6253,0.5793,0.5776,0.2688,0.2808,0.1845,0.1949
3,4,19718,4929,780,195,0.6195,0.6182,0.6195,0.6027,0.6024,0.2709,0.2727,0.1871,0.1878
4,5,19718,4929,780,195,0.6214,0.6202,0.6214,0.5989,0.5945,0.2670,0.2874,0.1841,0.1961



Spatial CV summary:


...


Full sample metrics:


,validation,n,ols_r2,ols_adj_r2,pseudo_r2_full,rmse_full_log,mae_full_log
0,full_sample,24647,0.6187,0.6177,0.6187,0.2703,0.186



Moran's I по остаткам полной модели:


,variable,weights,n,moran_I,expected_I,z_norm,p_norm,z_rand,p_rand,p_sim,permutations
0,residual_full,KNN_k8,24647,0.3124,-0.0,106.7,0.0,106.7159,0.0,0.001,999



Готово.
Все результаты сохранены в папку: model_6_diagnostics_new_zones


In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "model_6_diagnostics"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if "full_model" in globals():
    final_model = full_model
elif "model_full" in globals():
    final_model = model_full
elif "model" in globals():
    final_model = model
else:
    raise NameError("Не найдена финальная модель: full_model / model_full / model")

if "model_df" not in globals():
    raise NameError("Не найден model_df.")

if "MODEL_VARS" not in globals():
    raise NameError("Не найден MODEL_VARS.")

X_export = sm.add_constant(model_df[MODEL_VARS], has_constant="add")
y_export = model_df[DEPENDENT_VAR]

if "y_pred_full" not in model_df.columns:
    model_df["y_pred_full"] = final_model.predict(X_export)

if "residual_full" not in model_df.columns:
    model_df["residual_full"] = y_export - model_df["y_pred_full"]

if "abs_residual_full" not in model_df.columns:
    model_df["abs_residual_full"] = model_df["residual_full"].abs()


In [ ]:
coef_df = pd.DataFrame({
    "variable": final_model.params.index,
    "coef_log": final_model.params.values,
    "std_err": final_model.bse.values,
    "t_value": final_model.tvalues.values,
    "p_value": final_model.pvalues.values,
    "ci_low": final_model.conf_int()[0].values,
    "ci_high": final_model.conf_int()[1].values
})

coef_df["effect_percent"] = (np.exp(coef_df["coef_log"]) - 1) * 100
coef_df["ci_low_percent"] = (np.exp(coef_df["ci_low"]) - 1) * 100
coef_df["ci_high_percent"] = (np.exp(coef_df["ci_high"]) - 1) * 100

coef_df["abs_coef_log"] = coef_df["coef_log"].abs()
coef_df["significant_5pct"] = coef_df["p_value"] < 0.05
coef_df["significant_10pct"] = coef_df["p_value"] < 0.10

def classify_variable(var):
    if var == "const":
        return "constant"

    if var in ["zone_core", "zone_semi_periphery"]:
        return "zone_dummy"

    if var in component_vars:
        return "base_component"

    if "zone_core" in var and "_x_" in var:
        return "interaction_core"

    if "zone_semi_periphery" in var and "_x_" in var:
        return "interaction_semi_periphery"

    if "spatial_vars" in globals() and var in spatial_vars:
        return "spatial_control"

    return "other"

def extract_base_component(var):
    if "_x_zone_core" in var:
        return var.replace("_x_zone_core", "")

    if "_x_zone_semi_periphery" in var:
        return var.replace("_x_zone_semi_periphery", "")

    return var

coef_df["term_type"] = coef_df["variable"].apply(classify_variable)
coef_df["base_component"] = coef_df["variable"].apply(extract_base_component)

coef_df = coef_df.sort_values("abs_coef_log", ascending=False)

coef_path = os.path.join(OUTPUT_DIR, "model_6_coefficients.csv")

coef_df.to_csv(
    coef_path,
    index=False,
    encoding="utf-8-sig"
)

print("Файл с коэффициентами сохранён:", coef_path)

display(
    coef_df[
        [
            "variable",
            "term_type",
            "base_component",
            "coef_log",
            "effect_percent",
            "p_value",
            "significant_5pct",
            "ci_low",
            "ci_high"
        ]
    ].round(4)
)


Файл с коэффициентами сохранён: model_6_diagnostics_new_zones/model_6_coefficients.csv


,variable,term_type,base_component,coef_log,effect_percent,p_value,significant_5pct,ci_low,ci_high
0,const,constant,const,12.9403,4.167706e+07,0.0000,True,12.9292,12.9514
65,z_log_dist_to_core_center,spatial_control,z_log_dist_to_core_center,-0.1680,-1.546660e+01,0.0000,True,-0.1865,-0.1496
22,z_ln_total_area_x_zone_core,interaction_core,z_ln_total_area,0.0980,1.029100e+01,0.0000,True,0.0757,0.1202
6,z_building_newness,base_component,z_building_newness,0.0932,9.763500e+00,0.0000,True,0.0869,0.0994
34,z_mcd_proximity_x_zone_core,interaction_core,z_mcd_proximity,-0.0778,-7.485800e+00,0.0000,True,-0.0965,-0.0591
...,...,...,...,...,...,...,...,...,...
10,z_stop_proximity,base_component,z_stop_proximity,-0.0024,-2.375000e-01,0.3360,False,-0.0072,0.0025
16,z_green_proximity,base_component,z_green_proximity,0.0020,1.971000e-01,0.4402,False,-0.0030,0.0070
64,z_xy,spatial_control,z_xy,-0.0017,-1.747000e-01,0.2757,False,-0.0049,0.0014
3,z_ln_total_area,base_component,z_ln_total_area,0.0010,1.002000e-01,0.8633,False,-0.0104,0.0124


In [ ]:
if "OUTPUT_DIR" not in globals() or OUTPUT_DIR == "model_6_diagnostics":
    OUTPUT_DIR = "model_6_diagnostics_new_zones"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

base_export_cols = [
    DEPENDENT_VAR,
    "zone_name",
    "spatial_group",
    "y_pred_full",
    "residual_full",
    "abs_residual_full"
]

base_export_cols = [
    col for col in base_export_cols
    if col in model_df.columns
]

model_export_cols = [
    col for col in MODEL_VARS
    if col in model_df.columns
]

modeling_data = model_df[base_export_cols + model_export_cols].copy()

raw_feature_cols = [

    "hex_id",
    "hex_group_id",
    "x",
    "y",

    "price",
    "price_m2",
    "ln_price",
    "ln_price_m2",

    "total_area",
    "ln_total_area",
    "rooms_count",
    "rooms_count_num",
    "floor",
    "floor_comfort_score",

    "building_year",
    "building_age",
    "building_newness",
    "material_type",
    "material_score",

    "dist_nearest_metro_mcc",
    "dist_nearest_mcd",
    "dist_nearest_stop",
    "stops_500m",
    "metro_mcc_proximity",
    "mcd_proximity",
    "stop_proximity",
    "log_stops_500m",

    "schools_1000m",
    "kindergartens_1000m",
    "clinics_1000m",
    "log_schools_1000m",
    "log_kindergartens_1000m",
    "log_clinics_1000m",

    "green_area_1000m",
    "dist_nearest_green",
    "dist_nearest_water",
    "log_green_area_1000m",
    "green_proximity",
    "water_proximity",

    "major_road_300m",
    "railway_300m",
    "industrial_area_1000m",
    "log_industrial_area_1000m",
    "waste_1000m",

    "x_centered",
    "y_centered",
    "x2",
    "y2",
    "xy",
    "dist_to_core_center",
    "log_dist_to_core_center"
]

if "df" in globals():
    available_raw_cols = [
        col for col in raw_feature_cols
        if col in df.columns and col not in modeling_data.columns
    ]

    if len(available_raw_cols) > 0:
        raw_part = df.loc[modeling_data.index, available_raw_cols].copy()
        modeling_data = pd.concat([modeling_data, raw_part], axis=1)

if "geometry" in model_df.columns and "geometry_wkt" not in modeling_data.columns:
    modeling_data["geometry_wkt"] = model_df["geometry"].apply(
        lambda g: g.wkt if pd.notna(g) else np.nan
    )

modeling_data_path = os.path.join(OUTPUT_DIR, "model_6_modeling_data.csv")

modeling_data.to_csv(
    modeling_data_path,
    index=False,
    encoding="utf-8-sig"
)

print("Файл с таблицей модели сохранён:", modeling_data_path)
print("Размер таблицы:", modeling_data.shape)

display(modeling_data.head())


Файл с таблицей модели сохранён: model_6_diagnostics_new_zones/model_6_modeling_data.csv
Размер таблицы: (24647, 120)


,ln_price_m2,zone_name,spatial_group,y_pred_full,residual_full,abs_residual_full,zone_core,zone_semi_periphery,z_ln_total_area,z_rooms_count,...,log_industrial_area_1000m,waste_1000m,x_centered,y_centered,x2,y2,xy,dist_to_core_center,log_dist_to_core_center,geometry_wkt
0,11.374279,periphery,3135.0,12.382022,-1.007743,1.007743,0.0,0.0,-0.667346,-0.161762,...,11.421417,1,14899.115148,6867.946207,2.219836e+08,4.716869e+07,1.023263e+08,14165.172349,9.558612,POINT (426256.20864518086 6186032.815401154)
1,11.581918,periphery,3082.0,12.633750,-1.051832,1.051832,0.0,0.0,-0.838285,-0.161762,...,12.397745,0,14253.548553,-190.346170,2.031636e+08,3.623166e+04,-2.713108e+06,12467.320330,9.430946,POINT (425610.64204979566 6178974.523024487)
2,11.907217,periphery,3082.0,12.349321,-0.442103,0.442103,0.0,0.0,0.106421,1.147148,...,11.882750,0,14137.014966,-641.855718,1.998552e+08,4.119788e+05,-9.073924e+06,12414.101152,9.426669,POINT (425494.1084634445 6178523.013476344)
5,11.289782,periphery,3015.0,12.705798,-1.416016,1.416016,0.0,0.0,-0.655548,-0.161762,...,0.000000,0,13699.211578,4586.111933,1.876684e+08,2.103242e+07,6.282612e+07,12272.787732,9.415221,POINT (425056.3050755324 6183750.981127375)
6,12.500312,semi_periphery,2960.0,12.705694,-0.205382,0.205382,0.0,1.0,-2.939274,-1.470672,...,12.021625,0,12700.648708,-724.776329,1.613065e+08,5.253007e+05,-9.205130e+06,11012.578770,9.306884,POINT (424057.74220482836 6178440.092864576)


Аналитика

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

coef_path = Path("model_6_diagnostics_new_zones") / "model_6_coefficients.csv"
coef_df = pd.read_csv(coef_path)

coef_dict = dict(zip(coef_df["variable"], coef_df["coef_log"]))

base_vars = [

    "z_ln_total_area",
    "z_rooms_count",
    "z_floor_comfort_score",
    "z_building_newness",
    "z_material_score",

    "z_metro_mcc_proximity",
    "z_mcd_proximity",
    "z_stop_proximity",
    "z_log_stops_500m",

    "z_log_schools_1000m",
    "z_log_kindergartens_1000m",
    "z_log_clinics_1000m",
    "z_log_green_area_1000m",
    "z_green_proximity",
    "z_water_proximity",

    "z_major_road_300m",
    "z_railway_300m",
    "z_log_industrial_area_1000m",
    "z_waste_1000m",
]

def get_zone_coef(var, zone):
    """
    Возвращает коэффициент переменной в конкретной зоне.

    periphery:
        beta

    core:
        beta + beta_x_zone_core

    semi_periphery:
        beta + beta_x_zone_semi_periphery
    """

    base_coef = coef_dict.get(var, 0)

    if zone == "periphery":
        return base_coef

    elif zone == "core":
        interaction_name = f"{var}_x_zone_core"
        interaction_coef = coef_dict.get(interaction_name, 0)
        return base_coef + interaction_coef

    elif zone == "semi_periphery":
        interaction_name = f"{var}_x_zone_semi_periphery"
        interaction_coef = coef_dict.get(interaction_name, 0)
        return base_coef + interaction_coef

    else:
        raise ValueError("zone должна быть: periphery, core или semi_periphery")

rows = []

for var in base_vars:
    for zone in ["core", "semi_periphery", "periphery"]:
        beta_zone = get_zone_coef(var, zone)

        rows.append({
            "variable": var,
            "zone": zone,
            "coef_log": beta_zone,
            "effect_percent": (np.exp(beta_zone) - 1) * 100,
            "abs_coef_log": abs(beta_zone)
        })

zone_effects = pd.DataFrame(rows)

zone_effects = zone_effects.sort_values(
    ["zone", "abs_coef_log"],
    ascending=[True, False]
)

display(zone_effects.round(4))

zone_effects.to_csv(
    "model_6_zone_variable_effects.csv",
    index=False,
    encoding="utf-8-sig"
)


,variable,zone,coef_log,effect_percent,abs_coef_log
9,z_building_newness,core,0.1117,11.8149,0.1117
0,z_ln_total_area,core,0.0990,10.4015,0.0990
3,z_rooms_count,core,-0.0967,-9.2166,0.0967
18,z_mcd_proximity,core,-0.0940,-8.9702,0.0940
12,z_material_score,core,0.0820,8.5453,0.0820
42,z_water_proximity,core,0.0604,6.2282,0.0604
27,z_log_schools_1000m,core,0.0509,5.2186,0.0509
6,z_floor_comfort_score,core,0.0484,4.9635,0.0484
30,z_log_kindergartens_1000m,core,0.0475,4.8666,0.0475
51,z_log_industrial_area_1000m,core,-0.0401,-3.9281,0.0401


In [ ]:
feature_groups = {
    "Внутренние характеристики квартиры и дома": [
        "z_ln_total_area",
        "z_rooms_count",
        "z_floor_comfort_score",
        "z_building_newness",
        "z_material_score",
    ],

    "Транспортная доступность": [
        "z_metro_mcc_proximity",
        "z_mcd_proximity",
        "z_stop_proximity",
        "z_log_stops_500m",
    ],

    "Позитивные внешние характеристики": [
        "z_log_schools_1000m",
        "z_log_kindergartens_1000m",
        "z_log_clinics_1000m",
        "z_log_green_area_1000m",
        "z_green_proximity",
        "z_water_proximity",
    ],

    "Негативные внешние характеристики": [
        "z_major_road_300m",
        "z_railway_300m",
        "z_log_industrial_area_1000m",
        "z_waste_1000m",
    ],
}

def get_feature_group(var):
    for group_name, vars_list in feature_groups.items():
        if var in vars_list:
            return group_name
    return "Прочее"

zone_effects["feature_group"] = zone_effects["variable"].apply(get_feature_group)

zone_effects_grouped = zone_effects[
    zone_effects["feature_group"] != "Прочее"
].copy()

group_strength = (
    zone_effects_grouped
    .groupby(["zone", "feature_group"], as_index=False)
    .agg(
        block_strength_abs_beta=("abs_coef_log", "sum"),
        mean_abs_beta=("abs_coef_log", "mean"),
        n_features=("variable", "count")
    )
)

group_strength["block_share"] = (
    group_strength["block_strength_abs_beta"]
    / group_strength.groupby("zone")["block_strength_abs_beta"].transform("sum")
)

group_strength["block_share_percent"] = group_strength["block_share"] * 100

group_strength = group_strength.sort_values(
    ["zone", "block_strength_abs_beta"],
    ascending=[True, False]
)

display(group_strength.round(4))

group_strength.to_csv(
    "model_6_zone_block_strength.csv",
    index=False,
    encoding="utf-8-sig"
)


,zone,feature_group,block_strength_abs_beta,mean_abs_beta,n_features,block_share,block_share_percent
0,core,Внутренние характеристики квартиры и дома,0.4378,0.0876,5,0.5119,51.1944
2,core,Позитивные внешние характеристики,0.1900,0.0317,6,0.2222,22.2179
3,core,Транспортная доступность,0.1455,0.0364,4,0.1701,17.0129
1,core,Негативные внешние характеристики,0.0819,0.0205,4,0.0957,9.5748
4,periphery,Внутренние характеристики квартиры и дома,0.2212,0.0442,5,0.5181,51.8085
7,periphery,Транспортная доступность,0.0841,0.0210,4,0.1969,19.6908
6,periphery,Позитивные внешние характеристики,0.0714,0.0119,6,0.1672,16.7231
5,periphery,Негативные внешние характеристики,0.0503,0.0126,4,0.1178,11.7776
8,semi_periphery,Внутренние характеристики квартиры и дома,0.3456,0.0691,5,0.4976,49.7592
10,semi_periphery,Позитивные внешние характеристики,0.1399,0.0233,6,0.2014,20.1419


In [ ]:

pivot_strength = group_strength.pivot(
    index="feature_group",
    columns="zone",
    values="block_strength_abs_beta"
)

pivot_share = group_strength.pivot(
    index="feature_group",
    columns="zone",
    values="block_share_percent"
)

print("Сила блоков: сумма модулей коэффициентов")
display(pivot_strength.round(4))

print("Доля блоков внутри зоны, %")
display(pivot_share.round(1))

pivot_strength.to_csv(
    "model_6_zone_block_strength_pivot.csv",
    encoding="utf-8-sig"
)

pivot_share.to_csv(
    "model_6_zone_block_share_pivot.csv",
    encoding="utf-8-sig"
)


Сила блоков: сумма модулей коэффициентов


zone,core,periphery,semi_periphery
feature_group,,,
Внутренние характеристики квартиры и дома,0.4378,0.2212,0.3456
Негативные внешние характеристики,0.0819,0.0503,0.1054
Позитивные внешние характеристики,0.1900,0.0714,0.1399
Транспортная доступность,0.1455,0.0841,0.1037


Доля блоков внутри зоны, %


zone,core,periphery,semi_periphery
feature_group,,,
Внутренние характеристики квартиры и дома,51.2,51.8,49.8
Негативные внешние характеристики,9.6,11.8,15.2
Позитивные внешние характеристики,22.2,16.7,20.1
Транспортная доступность,17.0,19.7,14.9


In [ ]:
ranking = group_strength.copy()

ranking["rank_in_zone"] = (
    ranking
    .groupby("zone")["block_strength_abs_beta"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

ranking = ranking.sort_values(["zone", "rank_in_zone"])

display(
    ranking[
        [
            "zone",
            "rank_in_zone",
            "feature_group",
            "block_strength_abs_beta",
            "block_share_percent",
            "n_features"
        ]
    ].round(4)
)

ranking.to_csv(
    "model_6_zone_block_ranking.csv",
    index=False,
    encoding="utf-8-sig"
)


,zone,rank_in_zone,feature_group,block_strength_abs_beta,block_share_percent,n_features
0,core,1,Внутренние характеристики квартиры и дома,0.4378,51.1944,5
2,core,2,Позитивные внешние характеристики,0.1900,22.2179,6
3,core,3,Транспортная доступность,0.1455,17.0129,4
1,core,4,Негативные внешние характеристики,0.0819,9.5748,4
4,periphery,1,Внутренние характеристики квартиры и дома,0.2212,51.8085,5
7,periphery,2,Транспортная доступность,0.0841,19.6908,4
6,periphery,3,Позитивные внешние характеристики,0.0714,16.7231,6
5,periphery,4,Негативные внешние характеристики,0.0503,11.7776,4
8,semi_periphery,1,Внутренние характеристики квартиры и дома,0.3456,49.7592,5
10,semi_periphery,2,Позитивные внешние характеристики,0.1399,20.1419,6


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
from IPython.display import display, Markdown

DATA_PATH = Path("model_6_diagnostics_new_zones/model_6_modeling_data.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
elif "modeling_data" in globals():
    df = modeling_data.copy()
    print("Файл model_6_modeling_data.csv не найден")
else:
    raise FileNotFoundError(
        "Не найден model_6_diagnostics_new_zones/model_6_modeling_data.csv."
    )

OUTPUT_DIR = Path("zone_models_3_separate_no_interactions_linear_space")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_COL = "ln_price_m2"
ZONE_COL = "zone_name"

print("Размер данных для трех отдельных зональных моделей:", df.shape)
print("Зоны:")
display(df[ZONE_COL].value_counts())


Размер данных для трех отдельных зональных моделей: (24647, 120)
Зоны:


,count
zone_name,
periphery,11205
semi_periphery,9747
core,3695


In [ ]:
internal_vars = [
    "z_ln_total_area",
    "z_rooms_count",
    "z_floor_comfort_score",
    "z_building_newness",
    "z_material_score",
]

transport_vars = [
    "z_metro_mcc_proximity",
    "z_mcd_proximity",
    "z_stop_proximity",
    "z_log_stops_500m",
]

positive_external_vars = [
    "z_log_schools_1000m",
    "z_log_kindergartens_1000m",
    "z_log_clinics_1000m",
    "z_log_green_area_1000m",
    "z_green_proximity",
    "z_water_proximity",
]

negative_external_vars = [
    "z_major_road_300m",
    "z_railway_300m",
    "z_log_industrial_area_1000m",
    "z_waste_1000m",
]

spatial_vars = [
    "z_x_centered",
    "z_y_centered",
]

feature_groups = {
    "Внутренние характеристики квартиры и дома": internal_vars,
    "Транспортная доступность": transport_vars,
    "Позитивные внешние характеристики": positive_external_vars,
    "Негативные внешние характеристики": negative_external_vars,
}

MODEL_VARS = (
    internal_vars
    + transport_vars
    + positive_external_vars
    + negative_external_vars
    + spatial_vars
)

MODEL_VARS = [col for col in MODEL_VARS if col in df.columns]

print("Число признаков в отдельных зональных моделях без zone×feature и без x2/y2/xy:", len(MODEL_VARS))
print(MODEL_VARS)


Число признаков в отдельных зональных моделях без zone×feature и без x2/y2/xy: 21
['z_ln_total_area', 'z_rooms_count', 'z_floor_comfort_score', 'z_building_newness', 'z_material_score', 'z_metro_mcc_proximity', 'z_mcd_proximity', 'z_stop_proximity', 'z_log_stops_500m', 'z_log_schools_1000m', 'z_log_kindergartens_1000m', 'z_log_clinics_1000m', 'z_log_green_area_1000m', 'z_green_proximity', 'z_water_proximity', 'z_major_road_300m', 'z_railway_300m', 'z_log_industrial_area_1000m', 'z_waste_1000m', 'z_x_centered', 'z_y_centered']


In [ ]:
def regression_metrics(y_true, y_pred, n_params=None):
    residuals = y_true - y_pred

    sse = np.sum(residuals ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)

    r2 = 1 - sse / sst

    rmse = np.sqrt(np.mean(residuals ** 2))
    mae = np.mean(np.abs(residuals))

    if n_params is not None:
        n = len(y_true)
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_params - 1)
    else:
        adj_r2 = np.nan

    return {
        "r2": r2,
        "adj_r2": adj_r2,
        "rmse_log": rmse,
        "mae_log": mae,
        "mae_percent_approx": (np.exp(mae) - 1) * 100,
        "rmse_percent_approx": (np.exp(rmse) - 1) * 100,
    }

def get_feature_group(var):
    for group_name, vars_list in feature_groups.items():
        if var in vars_list:
            return group_name

    if var in spatial_vars:
        return "Пространственные контроли"

    if var == "const":
        return "Константа"

    return "Прочее"


In [ ]:
zone_order = ["core", "semi_periphery", "periphery"]

zone_names_ru = {
    "core": "центр",
    "semi_periphery": "полупериферия",
    "periphery": "периферия",
}

models = {}
metrics_rows = []
coef_rows = []
pred_rows = []

for zone in zone_order:

    zone_df = df[df[ZONE_COL] == zone].copy()

    needed_cols = [TARGET_COL] + MODEL_VARS
    zone_df = zone_df.dropna(subset=needed_cols).copy()

    if len(zone_df) < len(MODEL_VARS) + 5:
        print(f"Пропускаю зону {zone}: слишком мало наблюдений после dropna ({len(zone_df)}).")
        continue

    X = zone_df[MODEL_VARS].copy()
    X = sm.add_constant(X, has_constant="add")

    y = zone_df[TARGET_COL].copy()

    if "spatial_group" in zone_df.columns and zone_df["spatial_group"].nunique() >= 10:
        model = sm.OLS(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": zone_df["spatial_group"]}
        )
        se_type = "cluster_by_spatial_group"
    else:
        model = sm.OLS(y, X).fit(cov_type="HC3")
        se_type = "HC3"

    models[zone] = model

    y_pred = model.predict(X)
    residuals = y - y_pred

    m = regression_metrics(y, y_pred, n_params=len(MODEL_VARS))
    m.update({
        "zone": zone,
        "zone_ru": zone_names_ru[zone],
        "n": len(zone_df),
        "n_features": len(MODEL_VARS),
        "aic": model.aic,
        "bic": model.bic,
        "se_type": se_type,
    })

    metrics_rows.append(m)

    conf_int = model.conf_int()

    for var in model.params.index:
        beta = model.params[var]

        coef_rows.append({
            "zone": zone,
            "zone_ru": zone_names_ru[zone],
            "variable": var,
            "feature_group": get_feature_group(var),
            "coef_log": beta,
            "effect_percent": (np.exp(beta) - 1) * 100,
            "std_err": model.bse[var],
            "t_value": model.tvalues[var],
            "p_value": model.pvalues[var],
            "ci_low": conf_int.loc[var, 0],
            "ci_high": conf_int.loc[var, 1],
            "abs_coef_log": abs(beta),
            "significant_5pct": model.pvalues[var] < 0.05,
            "significant_10pct": model.pvalues[var] < 0.10,
        })

    temp_pred = zone_df[[ZONE_COL, TARGET_COL]].copy()
    temp_pred["zone"] = zone
    temp_pred["zone_ru"] = zone_names_ru[zone]
    temp_pred["y_pred_zone_model"] = y_pred
    temp_pred["residual_zone_model"] = residuals
    temp_pred["abs_residual_zone_model"] = np.abs(residuals)

    if "spatial_group" in zone_df.columns:
        temp_pred["spatial_group"] = zone_df["spatial_group"]

    pred_rows.append(temp_pred)

zone_metrics = pd.DataFrame(metrics_rows)
zone_coefficients = pd.DataFrame(coef_rows)
zone_predictions = pd.concat(pred_rows, axis=0, ignore_index=True) if pred_rows else pd.DataFrame()

zone_metrics.to_csv(
    OUTPUT_DIR / "zone_models_no_interactions_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

zone_coefficients.to_csv(
    OUTPUT_DIR / "zone_models_no_interactions_coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)

zone_predictions.to_csv(
    OUTPUT_DIR / "zone_models_no_interactions_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

display(zone_metrics.round(4))
display(
    zone_coefficients
    .sort_values(["zone", "abs_coef_log"], ascending=[True, False])
    .head(30)
    .round(4)
)


,r2,adj_r2,rmse_log,mae_log,mae_percent_approx,rmse_percent_approx,zone,zone_ru,n,n_features,aic,bic,se_type
0,0.5142,0.5115,0.3417,0.2394,27.0495,40.7308,core,центр,3695,21,2593.9497,2730.6739,cluster_by_spatial_group
1,0.4357,0.4344,0.2945,0.2126,23.6949,34.2426,semi_periphery,полупериферия,9747,21,3872.4150,4030.4787,cluster_by_spatial_group
2,0.3662,0.3650,0.2512,0.1761,19.2548,28.5550,periphery,периферия,11205,21,881.6698,1042.8003,cluster_by_spatial_group


,zone,zone_ru,variable,feature_group,coef_log,effect_percent,std_err,t_value,p_value,ci_low,ci_high,abs_coef_log,significant_5pct,significant_10pct
0,core,центр,const,Константа,13.2440,5.646501e+07,0.0387,342.2075,0.0000,13.1681,13.3198,13.2440,True,True
20,core,центр,z_x_centered,Пространственные контроли,-0.2125,-1.914070e+01,0.0428,-4.9588,0.0000,-0.2964,-0.1285,0.2125,True,True
21,core,центр,z_y_centered,Пространственные контроли,-0.2109,-1.901660e+01,0.0686,-3.0756,0.0021,-0.3453,-0.0765,0.2109,True,True
4,core,центр,z_building_newness,Внутренние характеристики квартиры и дома,0.1145,1.213500e+01,0.0110,10.4368,0.0000,0.0930,0.1360,0.1145,True,True
12,core,центр,z_log_clinics_1000m,Позитивные внешние характеристики,0.1138,1.205030e+01,0.0287,3.9635,0.0001,0.0575,0.1700,0.1138,True,True
1,core,центр,z_ln_total_area,Внутренние характеристики квартиры и дома,0.1121,1.186330e+01,0.0142,7.9097,0.0000,0.0843,0.1399,0.1121,True,True
2,core,центр,z_rooms_count,Внутренние характеристики квартиры и дома,-0.1104,-1.045370e+01,0.0149,-7.3987,0.0000,-0.1397,-0.0812,0.1104,True,True
10,core,центр,z_log_schools_1000m,Позитивные внешние характеристики,0.1062,1.120040e+01,0.0252,4.2083,0.0000,0.0567,0.1556,0.1062,True,True
5,core,центр,z_material_score,Внутренние характеристики квартиры и дома,0.0890,9.308100e+00,0.0139,6.4113,0.0000,0.0618,0.1162,0.0890,True,True
15,core,центр,z_water_proximity,Позитивные внешние характеристики,0.0544,5.585600e+00,0.0157,3.4607,0.0005,0.0236,0.0851,0.0544,True,True


In [ ]:
block_coef = zone_coefficients[
    zone_coefficients["feature_group"].isin(feature_groups.keys())
].copy()

block_strength = (
    block_coef
    .groupby(["zone", "zone_ru", "feature_group"], as_index=False)
    .agg(
        block_strength_abs_beta=("abs_coef_log", "sum"),
        mean_abs_beta=("abs_coef_log", "mean"),
        n_features=("variable", "count"),
        n_significant_5pct=("significant_5pct", "sum"),
        n_significant_10pct=("significant_10pct", "sum"),
    )
)

block_strength["block_share"] = (
    block_strength["block_strength_abs_beta"]
    / block_strength.groupby("zone")["block_strength_abs_beta"].transform("sum")
)

block_strength["block_share_percent"] = block_strength["block_share"] * 100

block_strength = block_strength.sort_values(
    ["zone", "block_strength_abs_beta"],
    ascending=[True, False]
)

block_strength.to_csv(
    OUTPUT_DIR / "zone_models_no_interactions_block_strength.csv",
    index=False,
    encoding="utf-8-sig"
)

display(block_strength.round(4))


,zone,zone_ru,feature_group,block_strength_abs_beta,mean_abs_beta,n_features,n_significant_5pct,n_significant_10pct,block_share,block_share_percent
0,core,центр,Внутренние характеристики квартиры и дома,0.4712,0.0942,5,5,5,0.4818,48.1786
2,core,центр,Позитивные внешние характеристики,0.3159,0.0526,6,4,4,0.3230,32.2993
1,core,центр,Негативные внешние характеристики,0.1107,0.0277,4,2,2,0.1132,11.3245
3,core,центр,Транспортная доступность,0.0802,0.0200,4,1,1,0.0820,8.1976
4,periphery,периферия,Внутренние характеристики квартиры и дома,0.2366,0.0473,5,4,4,0.5445,54.4523
7,periphery,периферия,Транспортная доступность,0.0922,0.0230,4,2,2,0.2122,21.2217
6,periphery,периферия,Позитивные внешние характеристики,0.0750,0.0125,6,2,2,0.1727,17.2686
5,periphery,периферия,Негативные внешние характеристики,0.0307,0.0077,4,0,1,0.0706,7.0573
8,semi_periphery,полупериферия,Внутренние характеристики квартиры и дома,0.3589,0.0718,5,5,5,0.4819,48.1888
10,semi_periphery,полупериферия,Позитивные внешние характеристики,0.1779,0.0297,6,4,4,0.2389,23.8939


In [ ]:
block_strength_pivot = block_strength.pivot(
    index="feature_group",
    columns="zone_ru",
    values="block_strength_abs_beta"
)

block_share_pivot = block_strength.pivot(
    index="feature_group",
    columns="zone_ru",
    values="block_share_percent"
)

ordered_cols = ["центр", "полупериферия", "периферия"]
block_strength_pivot = block_strength_pivot[[c for c in ordered_cols if c in block_strength_pivot.columns]]
block_share_pivot = block_share_pivot[[c for c in ordered_cols if c in block_share_pivot.columns]]

print("Сила блоков: сумма |β|")
display(block_strength_pivot.round(3))

print("Доля блоков внутри зоны, %")
display(block_share_pivot.round(1))

block_strength_pivot.to_csv(
    OUTPUT_DIR / "zone_models_no_interactions_block_strength_pivot.csv",
    encoding="utf-8-sig"
)

block_share_pivot.to_csv(
    OUTPUT_DIR / "zone_models_no_interactions_block_share_pivot.csv",
    encoding="utf-8-sig"
)


Сила блоков: сумма |β|


zone_ru,центр,полупериферия,периферия
feature_group,,,
Внутренние характеристики квартиры и дома,0.471,0.359,0.237
Негативные внешние характеристики,0.111,0.081,0.031
Позитивные внешние характеристики,0.316,0.178,0.075
Транспортная доступность,0.080,0.127,0.092


Доля блоков внутри зоны, %


zone_ru,центр,полупериферия,периферия
feature_group,,,
Внутренние характеристики квартиры и дома,48.2,48.2,54.5
Негативные внешние характеристики,11.3,10.8,7.1
Позитивные внешние характеристики,32.3,23.9,17.3
Транспортная доступность,8.2,17.1,21.2


In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from IPython.display import display, Markdown
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ZONE_ORDER = ["periphery", "semi_periphery", "core"]
ZONE_NAMES_RU = {
    "core": "центр",
    "semi_periphery": "полупериферия",
    "periphery": "периферия",
}

MODEL_1_DIR = Path("model_6_diagnostics_new_zones")
MODEL_2_DIR = Path("zone_models_3_separate_no_interactions_linear_space")
MODEL_3_DIR = Path("model_7_no_internal_2rooms_middle_floor_general")
MODEL_4_DIR = Path("model_8_no_internal_2rooms_middle_floor_by_zone")

for path in [MODEL_1_DIR, MODEL_2_DIR, MODEL_3_DIR, MODEL_4_DIR]:
    path.mkdir(exist_ok=True)

def percent_from_log_coef(beta):
    if pd.isna(beta):
        return np.nan
    return (np.exp(beta) - 1) * 100

def rmse_local(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def regression_metrics_local(y_true, y_pred, n_params=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    residuals = y_true - y_pred
    sse = np.sum(residuals ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - sse / sst if sst != 0 else np.nan
    rmse_value = np.sqrt(np.mean(residuals ** 2))
    mae_value = np.mean(np.abs(residuals))

    if n_params is not None and len(y_true) > n_params + 1:
        adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / (len(y_true) - n_params - 1)
    else:
        adj_r2 = np.nan

    return {
        "r2": r2,
        "adj_r2": adj_r2,
        "rmse_log": rmse_value,
        "mae_log": mae_value,
        "rmse_percent_approx": percent_from_log_coef(rmse_value),
        "mae_percent_approx": percent_from_log_coef(mae_value),
    }

def coefficient_table_from_model(model):
    conf_int = model.conf_int()
    out = pd.DataFrame({
        "variable": model.params.index,
        "coef_log": model.params.values,
        "std_err": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "ci_low": conf_int[0].values,
        "ci_high": conf_int[1].values,
    })
    out["effect_percent"] = percent_from_log_coef(out["coef_log"])
    out["ci_low_percent"] = percent_from_log_coef(out["ci_low"])
    out["ci_high_percent"] = percent_from_log_coef(out["ci_high"])
    out["abs_coef_log"] = out["coef_log"].abs()
    out["significant_5pct"] = out["p_value"] < 0.05
    out["significant_10pct"] = out["p_value"] < 0.10
    return out

def general_model_zone_effect_table(model, model_name, model_description, output_dir, filename):
    params = model.params
    bse = model.bse
    pvalues = model.pvalues
    conf_int = model.conf_int()

    rows = []
    zone_to_var = {
        "periphery": None,
        "semi_periphery": "zone_semi_periphery",
        "core": "zone_core",
    }

    for zone in ZONE_ORDER:
        var = zone_to_var[zone]
        if var is None:
            beta = 0.0
            std_err = np.nan
            p_value = np.nan
            ci_low = 0.0
            ci_high = 0.0
            note = "Базовая зона сравнения"
        else:
            beta = params.get(var, np.nan)
            std_err = bse.get(var, np.nan)
            p_value = pvalues.get(var, np.nan)
            if var in conf_int.index:
                ci_low = conf_int.loc[var, 0]
                ci_high = conf_int.loc[var, 1]
            else:
                ci_low = np.nan
                ci_high = np.nan
            note = "Эффект зоны относительно периферии при средних значениях стандартизированных признаков"

        rows.append({
            "model": model_name,
            "model_description": model_description,
            "zone": zone,
            "zone_ru": ZONE_NAMES_RU[zone],
            "baseline_zone": "periphery",
            "zone_term": var if var is not None else "baseline",
            "coef_log_relative_to_periphery": beta,
            "effect_percent_relative_to_periphery": percent_from_log_coef(beta),
            "std_err": std_err,
            "p_value": p_value,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "ci_low_percent": percent_from_log_coef(ci_low),
            "ci_high_percent": percent_from_log_coef(ci_high),
            "interpretation_note": note,
        })

    table = pd.DataFrame(rows)
    table.to_csv(output_dir / filename, index=False, encoding="utf-8-sig")
    return table

def separate_models_zone_effect_table(models_dict, metrics_df, model_name, model_description, output_dir, filename):
    rows = []
    intercepts = {}

    for zone, model in models_dict.items():
        intercepts[zone] = model.params.get("const", np.nan)

    baseline = intercepts.get("periphery", np.nan)

    n_by_zone = {}
    if isinstance(metrics_df, pd.DataFrame) and "zone" in metrics_df.columns:
        n_col = "n" if "n" in metrics_df.columns else None
        if n_col is not None:
            n_by_zone = dict(zip(metrics_df["zone"], metrics_df[n_col]))

    for zone in ZONE_ORDER:
        intercept = intercepts.get(zone, np.nan)
        beta = intercept - baseline if pd.notna(intercept) and pd.notna(baseline) else np.nan
        rows.append({
            "model": model_name,
            "model_description": model_description,
            "zone": zone,
            "zone_ru": ZONE_NAMES_RU[zone],
            "baseline_zone": "periphery",
            "coef_log_relative_to_periphery": beta,
            "effect_percent_relative_to_periphery": percent_from_log_coef(beta),
            "intercept_log_price_m2": intercept,
            "expected_price_m2_at_reference": np.exp(intercept) if pd.notna(intercept) else np.nan,
            "n": n_by_zone.get(zone, np.nan),
            "interpretation_note": "Разница констант отдельных зональных моделей относительно периферии",
        })

    table = pd.DataFrame(rows)
    table.to_csv(output_dir / filename, index=False, encoding="utf-8-sig")
    return table

In [ ]:
DATA_PATH = MODEL_1_DIR / "model_6_modeling_data.csv"

if DATA_PATH.exists():
    analysis_df = pd.read_csv(DATA_PATH)
elif "df" in globals():
    analysis_df = df.copy()
else:
    raise FileNotFoundError("Не найден model_6_modeling_data.csv и переменная df.")

analysis_df = analysis_df.loc[:, ~analysis_df.columns.duplicated()].copy()

def coefficient_table_from_model(model):
    conf = np.asarray(model.conf_int())

    result = pd.DataFrame({
        "feature": model.params.index.to_numpy(),
        "coef_log": model.params.to_numpy(),
        "std_error": model.bse.to_numpy(),
        "t_value": model.tvalues.to_numpy(),
        "p_value": model.pvalues.to_numpy(),
        "ci_low": conf[:, 0],
        "ci_high": conf[:, 1],
    })

    result["coef_percent"] = np.where(
        result["feature"] == "const",
        np.nan,
        (np.exp(result["coef_log"]) - 1) * 100
    )

    result["abs_coef_log"] = result["coef_log"].abs()
    result["abs_coef_percent"] = result["coef_percent"].abs()

    result["significance"] = ""
    result.loc[result["p_value"] < 0.1, "significance"] = "*"
    result.loc[result["p_value"] < 0.05, "significance"] = "**"
    result.loc[result["p_value"] < 0.01, "significance"] = "***"

    result["significant_10pct"] = result["p_value"] < 0.1
    result["significant_5pct"] = result["p_value"] < 0.05
    result["significant_1pct"] = result["p_value"] < 0.01

    return result

TARGET_COL = "ln_price_m2"
ZONE_COL = "zone_name"

if "zone_core" not in analysis_df.columns:
    analysis_df["zone_core"] = (analysis_df[ZONE_COL] == "core").astype(int)

if "zone_semi_periphery" not in analysis_df.columns:
    analysis_df["zone_semi_periphery"] = (analysis_df[ZONE_COL] == "semi_periphery").astype(int)

if "rooms_count_num" not in analysis_df.columns and "rooms_count" in analysis_df.columns:
    analysis_df["rooms_count_num"] = pd.to_numeric(analysis_df["rooms_count"], errors="coerce")

if "floor_comfort_score" not in analysis_df.columns:
    if {"is_first_floor", "is_last_floor", "is_middle_floor"}.issubset(analysis_df.columns):
        analysis_df["floor_comfort_score"] = np.nan
        analysis_df.loc[analysis_df["is_first_floor"] == 1, "floor_comfort_score"] = 0
        analysis_df.loc[analysis_df["is_last_floor"] == 1, "floor_comfort_score"] = 0.5
        analysis_df.loc[analysis_df["is_middle_floor"] == 1, "floor_comfort_score"] = 1
    else:
        raise ValueError("Для фильтра не найден floor_comfort_score или признаки is_first_floor/is_last_floor/is_middle_floor.")

analysis_df["rooms_count_num"] = pd.to_numeric(analysis_df["rooms_count_num"], errors="coerce")
analysis_df["floor_comfort_score"] = pd.to_numeric(analysis_df["floor_comfort_score"], errors="coerce")

model_3_df = analysis_df[
    (analysis_df["rooms_count_num"] == 2)
    & (analysis_df["floor_comfort_score"] >= 0.999)
    & (analysis_df[ZONE_COL].isin(["core", "semi_periphery", "periphery"]))
].copy()

transport_vars_no_internal = [
    "z_metro_mcc_proximity",
    "z_mcd_proximity",
    "z_stop_proximity",
    "z_log_stops_500m",
]

positive_external_vars_no_internal = [
    "z_log_schools_1000m",
    "z_log_kindergartens_1000m",
    "z_log_clinics_1000m",
    "z_log_green_area_1000m",
    "z_green_proximity",
    "z_water_proximity",
]

negative_external_vars_no_internal = [
    "z_major_road_300m",
    "z_railway_300m",
    "z_log_industrial_area_1000m",
    "z_waste_1000m",
]

external_component_vars = (
    transport_vars_no_internal
    + positive_external_vars_no_internal
    + negative_external_vars_no_internal
)

external_component_vars = [col for col in external_component_vars if col in model_3_df.columns]

zone_vars_general = [
    "zone_core",
    "zone_semi_periphery",
]

interaction_vars_no_internal = []

for component in external_component_vars:
    for zone_var in zone_vars_general:
        new_col = f"{component}_x_{zone_var}"
        model_3_df[new_col] = (
            pd.to_numeric(model_3_df[component], errors="coerce")
            * pd.to_numeric(model_3_df[zone_var], errors="coerce")
        )
        interaction_vars_no_internal.append(new_col)

spatial_vars_general = [
    "z_x_centered",
    "z_y_centered",
    "z_x2",
    "z_y2",
    "z_xy",
    "z_log_dist_to_core_center",
]

spatial_vars_general = [col for col in spatial_vars_general if col in model_3_df.columns]

MODEL_3_VARS = (
    zone_vars_general
    + external_component_vars
    + interaction_vars_no_internal
    + spatial_vars_general
)

MODEL_3_VARS = list(dict.fromkeys(MODEL_3_VARS))

for col in [TARGET_COL] + MODEL_3_VARS:
    model_3_df[col] = pd.to_numeric(model_3_df[col], errors="coerce")

for col in MODEL_3_VARS:
    model_3_df[col] = model_3_df[col].fillna(model_3_df[col].median())

model_3_df = model_3_df.dropna(subset=[TARGET_COL, ZONE_COL] + MODEL_3_VARS).copy()

print("Модель 3: общая модель без внутренних характеристик")
print("Фильтр: только 2-комнатные, не первый и не последний этаж")
print("N:", len(model_3_df))
print("Распределение по зонам:")
display(model_3_df[ZONE_COL].value_counts())
print("Число признаков:", len(MODEL_3_VARS))

X_model_3 = sm.add_constant(model_3_df[MODEL_3_VARS], has_constant="add")
y_model_3 = model_3_df[TARGET_COL]

model_3_general_no_internal = sm.OLS(y_model_3, X_model_3).fit(cov_type="HC3")

y_model_3_pred = model_3_general_no_internal.predict(X_model_3)

model_3_metrics = pd.DataFrame([{
    "model": "model_3_general_no_internal_2rooms_middle_floor",
    "sample": "2 rooms, not first floor, not last floor",
    "n": len(model_3_df),
    "n_features": len(MODEL_3_VARS),
    **regression_metrics_local(
        y_model_3,
        y_model_3_pred,
        n_params=len(MODEL_3_VARS)
    ),
    "aic": model_3_general_no_internal.aic,
    "bic": model_3_general_no_internal.bic,
}])

model_3_coefficients = coefficient_table_from_model(model_3_general_no_internal)

model_3_metrics.to_csv(
    MODEL_3_DIR / "model_3_general_no_internal_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

model_3_coefficients.to_csv(
    MODEL_3_DIR / "model_3_general_no_internal_coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)

model_3_df[[TARGET_COL, ZONE_COL] + MODEL_3_VARS].to_csv(
    MODEL_3_DIR / "model_3_general_no_internal_modeling_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print(model_3_general_no_internal.summary())
display(model_3_metrics.round(4))
display(
    model_3_coefficients
    .sort_values("abs_coef_log", ascending=False)
    .head(30)
    .round(4)
)

Модель 3: общая модель без внутренних характеристик
Фильтр: только 2-комнатные, не первый и не последний этаж
N: 8424
Распределение по зонам:


,count
zone_name,
periphery,3701
semi_periphery,3419
core,1304


Число признаков: 50
                            OLS Regression Results                            
Dep. Variable:            ln_price_m2   R-squared:                       0.580
Model:                            OLS   Adj. R-squared:                  0.578
Method:                 Least Squares   F-statistic:                     200.9
Date:                Sun, 07 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:06:27   Log-Likelihood:                -699.63
No. Observations:                8424   AIC:                             1501.
Df Residuals:                    8373   BIC:                             1860.
Df Model:                          50                                         
Covariance Type:                  HC3                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------

,model,sample,n,n_features,r2,adj_r2,rmse_log,mae_log,rmse_percent_approx,mae_percent_approx,aic,bic
0,model_3_general_no_internal_2rooms_middle_floor,"2 rooms, not first floor, not last floor",8424,50,0.5805,0.578,0.2629,0.1931,30.0729,21.2992,1501.2586,1860.2395


,feature,coef_log,std_error,t_value,p_value,ci_low,ci_high,coef_percent,abs_coef_log,abs_coef_percent,significance,significant_10pct,significant_5pct,significant_1pct
0,const,13.0041,0.0096,1351.0304,0.0000,12.9852,13.0229,NaN,13.0041,NaN,***,True,True,True
50,z_log_dist_to_core_center,-0.2693,0.0169,-15.9737,0.0000,-0.3023,-0.2362,-23.6049,0.2693,23.6049,***,True,True,True
19,z_mcd_proximity_x_zone_core,-0.1150,0.0149,-7.7061,0.0000,-0.1442,-0.0857,-10.8604,0.1150,10.8604,***,True,True,True
29,z_log_clinics_1000m_x_zone_core,-0.0849,0.0206,-4.1124,0.0000,-0.1254,-0.0444,-8.1401,0.0849,8.1401,***,True,True,True
45,z_x_centered,-0.0815,0.0035,-23.3838,0.0000,-0.0883,-0.0747,-7.8283,0.0815,7.8283,***,True,True,True
28,z_log_kindergartens_1000m_x_zone_semi_periphery,-0.0772,0.0091,-8.4959,0.0000,-0.0950,-0.0594,-7.4255,0.0772,7.4255,***,True,True,True
26,z_log_schools_1000m_x_zone_semi_periphery,0.0763,0.0086,8.8524,0.0000,0.0594,0.0932,7.9328,0.0763,7.9328,***,True,True,True
25,z_log_schools_1000m_x_zone_core,0.0763,0.0167,4.5586,0.0000,0.0435,0.1091,7.9314,0.0763,7.9314,***,True,True,True
7,z_log_schools_1000m,-0.0627,0.0057,-10.9694,0.0000,-0.0739,-0.0515,-6.0811,0.0627,6.0811,***,True,True,True
3,z_metro_mcc_proximity,0.0513,0.0044,11.7384,0.0000,0.0427,0.0598,5.2598,0.0513,5.2598,***,True,True,True


In [ ]:
all_zone_effects = pd.concat(
    [
        model_1_zone_effect,
        model_2_zone_effect,
        model_3_zone_effect,
        model_4_zone_effect,
    ],
    axis=0,
    ignore_index=True,
)

all_zone_effects_out = all_zone_effects[[
    "model",
    "model_description",
    "zone_ru",
    "baseline_zone",
    "coef_log_relative_to_periphery",
    "effect_percent_relative_to_periphery",
    "interpretation_note",
]].copy()

all_zone_effects_out = all_zone_effects_out.rename(columns={
    "model": "Модель",
    "model_description": "Описание модели",
    "zone_ru": "Зона",
    "baseline_zone": "Базовая зона сравнения",
    "coef_log_relative_to_periphery": "Коэффициент зоны в лог-модели / разница констант",
    "effect_percent_relative_to_periphery": "Эффект зоны к периферии, %",
    "interpretation_note": "Как считать",
})

all_zone_effects_out.to_csv(
    "all_4_models_zone_effects_coefficients_percent.csv",
    index=False,
    encoding="utf-8-sig",
)

display(all_zone_effects_out.round(4))
print("Сводная таблица сохранена: all_4_models_zone_effects_coefficients_percent.csv")

,Модель,Описание модели,Зона,Базовая зона сравнения,Коэффициент зоны в лог-модели / разница констант,"Эффект зоны к периферии, %",Как считать
0,model_1_general_full_with_interactions,"Общая модель с внутренними характеристиками, з...",периферия,periphery,0.0000,0.0000,Базовая зона сравнения
1,model_1_general_full_with_interactions,"Общая модель с внутренними характеристиками, з...",полупериферия,periphery,-0.0028,-0.2840,Эффект зоны относительно периферии при средних...
2,model_1_general_full_with_interactions,"Общая модель с внутренними характеристиками, з...",центр,periphery,0.0205,2.0716,Эффект зоны относительно периферии при средних...
3,model_2_three_zone_models_full,Три отдельные зональные модели с внутренними х...,периферия,periphery,0.0000,0.0000,Разница констант отдельных зональных моделей о...
4,model_2_three_zone_models_full,Три отдельные зональные модели с внутренними х...,полупериферия,periphery,0.2300,25.8538,Разница констант отдельных зональных моделей о...
5,model_2_three_zone_models_full,Три отдельные зональные модели с внутренними х...,центр,periphery,0.4585,58.1639,Разница констант отдельных зональных моделей о...
6,model_3_general_no_internal_2rooms_middle_floor,Общая модель без внутренних характеристик; тол...,периферия,periphery,0.0000,0.0000,Базовая зона сравнения
7,model_3_general_no_internal_2rooms_middle_floor,Общая модель без внутренних характеристик; тол...,полупериферия,periphery,-0.0246,-2.4334,Эффект зоны относительно периферии при средних...
8,model_3_general_no_internal_2rooms_middle_floor,Общая модель без внутренних характеристик; тол...,центр,periphery,-0.0153,-1.5136,Эффект зоны относительно периферии при средних...
9,model_4_three_zone_models_no_internal_2rooms_m...,Три отдельные зональные модели без внутренних ...,периферия,periphery,0.0000,0.0000,Разница констант отдельных зональных моделей о...


Сводная таблица сохранена: all_4_models_zone_effects_coefficients_percent.csv
